# Ligand PathFinder — peptide (un)binding trajectories · Google Colab

Builds physically reasonable **peptide binding/unbinding pathways** for a
protein–peptide complex, and a Rosetta binding-energy estimate at every step
along them.

Method: Zlobin A, Eisenhuth P, Ferrario V, Meiler J. *Exploration of ligand (un)binding trajectories with Rosetta*. bioRxiv 2026. doi:[10.64898/2026.10.07.757317](https://www.biorxiv.org/content/10.64898/2026.10.07.757317v1)
Code: [`meilerlab/LigandPathFinder`](https://github.com/meilerlab/LigandPathFinder) — run here **unmodified**; this
notebook prepares its inputs, launches it, and analyses what it writes.

> **Google Colab.** Rosetta comes from `pip install pyrosetta` (step 2, the default here) through a small `rosetta_scripts`-compatible front end, since LPF execs that binary rather than importing Rosetta. A Drive-hosted Rosetta binary also works and skips the per-session install.

> Runs outlast Colab sessions. Put `output.path` on Drive and leave checkpointing on; re-running the trajectory cell resumes where it stopped.

## The two stages

| Stage | Script | In | Out |
|---|---|---|---|
| 1 · coarse path | `lpf_coarse_path.py` | start pose + a rough exit direction (2 points) | `coarsepath.txt` — a guide path found by recursive bisection |
| 2 · trajectory | `lpf_trajectory.py` | start pose + the full guide path | per-window structures + **`interface_scores.txt`** |

Stage 1 is stochastic; the recommended practice is several replicates, then the
medoid path forward. You can skip it entirely and supply a path from CAVER or
AQUA-DUCT.

## What you need

1. **Rosetta.** Either a `rosetta_scripts` binary plus its database, or
   PyRosetta via `pip` — step 2 sets up whichever you pick, and the rest of the
   notebook is identical. Free for academic and non-commercial use, and
   separately licensed from LPF:
   [downloads.rosettacommons.org](https://downloads.rosettacommons.org/downloads/academic/),
   [github.com/RosettaCommons/rosetta](https://github.com/RosettaCommons/rosetta).
2. **A protein–peptide complex in a functionally relevant bound pose.** LPF does
   *not* find the binding pose, and nothing here will either — dock or predict
   it first (Rosetta, Vina, GLIDE, AF3/Boltz/chai-1), and look at it.
3. Patience. One trajectory window is one Rosetta run of `nstruct` decoys; a
   full path is tens of windows. Expect tens of minutes to many hours.

## What this notebook will not do for you

* find or validate a binding pose,
* decide whether the exit route it proposes is the biologically relevant one,
* turn a single noisy replicate into a converged energy profile — pool
  replicates for that (appendix).

## How a peptide is handled here

LPF is built around a single Rosetta ligand residue: it reads `NBR_ATOM` from
`lig.params`, tracks that one atom along the path, restrains every ligand atom
to a per-atom ghost residue, and draws conformers from `lig_conformers.pdb`.
Every constraint it writes is of the form `<atom> <resnum><chain>` — one residue.

So this notebook treats the **whole peptide as one large flexible ligand**:

| | |
|---|---|
| representation | the peptide becomes a single Rosetta residue type via `molfile_to_params.py` |
| in `start.pdb` | HETATM records, one residue, one chain |
| flexibility | entirely from the conformer library — the options file sets `-packing:ignore_ligand_chi true`, so Rosetta does not sample those torsions on the fly |
| tracked atom | re-picked to a central backbone atom (`molfile_to_params` tends to choose a terminal one) |
| protocols | the upstream XMLs, retuned for peptide size: scoring grid sized from the molecule, wider interface cutoffs, looser C-alpha restraints, bigger `Transform` box |

**This keeps the upstream code unmodified.** The alternative — keeping the
peptide as a real polypeptide chain — would need `util.py` forked
(`read_window` only sees HETATM; `create_RMSD_constraints` and
`swap_lig_coords` key atoms by name alone, which collides across residues) and
the RosettaLigand movers replaced. That is a different project.

**Practical ceiling: roughly 2-12 residues.** Beyond that `molfile_to_params`
struggles, the scoring grid gets expensive, and a conformer library stops being
able to represent the accessible space.

**The peptide parameter defaults in this notebook are not from the paper.** They
are the authors' small-molecule values scaled for a molecule an order of
magnitude larger, with the reasoning written next to each one. Treat them as a
starting point and retune using the checklist in the appendix.

---
## 0 · Environment

Installs the Python side and mounts Drive. Rosetta itself comes in the next step.

In [ ]:
#@title Install packages and mount Google Drive {display-mode: "form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}

import os, sys, subprocess
print("installing numpy, scipy, matplotlib, rdkit, py3Dmol ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "scipy", "matplotlib", "rdkit", "py3Dmol"], check=True)
# gzip is called after every Rosetta call; zip only when output.archive is on
print("installing zip ...")
subprocess.run("apt-get -qq install -y zip > /dev/null", shell=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("\nDrive mounted at /content/drive — keep the Rosetta binary and the run "
          "outputs here so a dropped session does not lose them.")

print("\nCPUs visible to this runtime:", os.cpu_count())
print("Note: MPI (rosetta_scripts.n_proc > 1) needs an .mpi. Rosetta build AND "
      "openmpi installed. A standard Colab runtime is better served by n_proc=1.")

In [ ]:
#@title Helper module `lpf_nb.py` — run once, then collapse {display-mode: "form"}
# This writes the notebook's helper module to disk and imports it.
# It is the same file that ships with the local notebooks; nothing here
# patches LigandPathFinder.
# Imports os/sys here as well as in the install cell above, so this
# notebook still works if you skip the install cell on a warm runtime.
import base64, importlib, os, sys, zlib

_LPF_NB_B64 = (
    "..."
    "jQ=="
)

# the rosetta_scripts-compatible front end for the PyRosetta backend
_SHIM_B64 = (
    "eNrdfXt320b22P/8FLNIuwJiEpa8ybahQ58otpx1Gz8qKdlttToQSIAiLBBgAdAUV8ue34f4fcJ+kt7HPAFQkrPebU99TiIS"
    "nLkzc+fOfc/FV797uq6rp9OseJoWn8Rq2yzK4g8Dz/MGx+KqKuu0aeKonlXZqqmvRrNyuYqbbJqnYl6VRSPSIhHTeHaTwp+t"
    "+LA95R7hYPBzdh3Djx/iZvE6K5K0EkmZ1qIoG5EtV2XVCNlW/O9/+3eRNaJaF7VoFml32MHX6W06WzcxjPu1iGsRi3o9XVXl"
    "LK3hC4wC3QpRpXFSi6t6VlZpWM+uRLluRDknmJuyusmK60GSVemsKaut8Os0Fesmy0MYOJJDBqFZg6gX2QonDIPNFsJMYSjq"
    "cnC1ylYiK+omznNAm+wPYxawllqUm8Ks9zr7lIqfP7yGqQJ+YRq43FlcwH95Dqg6X2S14MWKWQ6QEA8x9ItXY2y6qtI6LRrG"
    "Tp1ns1Qtq4sq2KElYiTPipSh4MBrgDk0GJYrPONO4i9vf4bHVbm+XpjlDxGxg02VNTCdJJ2VsL2vfmR0A24QyWKeASFkBcGc"
    "l9WSRzuoxRn+/CGuatj29HYFKK8Bs2VWNAOatifn7V2Jm3SLq4G5hR9rwB7AwCZ5vC5mC+i+qeLVCv7Sfo1GNKGRfBjQHAHF"
    "jNSC1jpbxMV1WgNeX9q4WCMN1k1ZJsJPb+NZk2/FRuFnus7ypA7Gg4GAfy2kih/KVZOVgLsR7He1njXinRgBcY3hLCzGq2Qq"
    "wqewResc2v6VIOC/0YrWPwY6bcpZmYsmrm/C22UuRlkxRtSNa34GEB4Y13QAggNc4pjtGdAv9vj4O/fC3aC94t71rDs5/oVm"
    "pxc5GJx8SuGozPP4mg9uwSSVZ3XDx7aY5esEkX/1Aw4A9E9TJsqouUktYEOSNBkA2Di7XsD+lubEhFmRNX6AZ4r2XS2Zpkvb"
    "Wa4yAAWkU4o0hoPYPcowBC2HeJAk3wM+g3I69bZu0qWoAFtASrAG5BYj4EN5uoSTBZOD5ymQzMktTBKOCsDQRwKnBcwxA4aH"
    "jcU0ncVwoOTsiqYqcz5W9tmTpARtorM/vXkb/Zf3P55NvqGNAVIXCsfxDPBQi29oVTA1ydRgwX6SzmMgKXEUhHpP2/9OFEKg"
    "KyIyi2FrEFuSBVmnGfA7XSfXabMX2P86Gj0TP/2Ip/H0+K3AQ/exnIbuOn49Of3x/dnJ5EjAyU1XZgSJcEDfDDoC7a3WDSD0"
    "LM3nIzjIs5shHLcMplukaUJsVXeVuGLhY0gjap2HcLUFBlAjQOBJzWDwKouvi7JG8oMNBV46+ltalUgqaTWHaURE+HXY3DbC"
    "52H1mOqszxLxPWzIi6cbEFLlJjo8PPz2M2YD+zVNzclWxxl+UYQs/w4Gbz+8GcOyizQU/xUxdxUuV1noSCnF9A6Y/ot4mRK/"
    "RUrbZCBoFmWeDNoTKSKkGmScR3gAYO8BRc+FJFFDfiSuQEbCrhghB2fHZvj4fVbCqefDtC3XlZbTKHiKGYg/ZAXTrZTwRsAf"
    "AM9FxWEAqsFSRNF83ayrNIqUvI8L6BhLbKhn1TUdXfW9rNUnOLLqI9EUahmDQRTB3GoAAWAnwjsKD2G8r1jeCNj2ZU3IAvUA"
    "O8EpmKZz/KmKC2QaLMhqxW22YrkGToayJK5AsQBIwAvXS9i29BNoFBtUK2KhmWSexp9IOAMvAXQjtUEXn7SI6OX7d2fnp8dv"
    "3p3Dx5/PxFMRvXndfgpn2XpyfnL69gzW4XtxUy6jVZxVEXAUYpRF4w2FB6IsT1vPkmyRJlWc24+DwenJf/vlzenJKxzml7fv"
    "GGwDCM/5GGBPczKSNAcK+gv0Q/SNvtg/wqDhgl8W9uCrMcki3AFLYwIsrJdp/Rz3rJIKVpoD9V+jAtYjbAa0K7+8BWT9+c35"
    "n6Jfj3/+5QTwdUcH31MCEDHmCFn9wJWq9LglT72hhKUltwtNbwk/QkC3+mu9ntPXwQ52B8QASzfNgODMfPLxf8DDaBA4c/j3"
    "bJWDshh31AhHJSQ5SnrPBngLSCBUueBHeqROOkgJ1J8QqHok8R7foFaZZpX4FOdrQC9o4/Myz8sNYr0pb9KCTxf3YHZC+5MQ"
    "uKYE2KTWZc2CxBTBeQ7t4XCRJnHww4FAzoeqpq1OKPlB0h5EpAQohfQa1gVqOrAtOO961rj5iLvQQVSJWIHt9tRGj8UR4B5Q"
    "H+HmRLjVY+GFuCFml8fyC6hPcnfVP0/v+li8AwZPJ22F0AiQfKTB897L521ARAgwFI3NdIDfdiyTYN1qwRNxcUkPM/h4SJ8A"
    "P6iPi+8Bl4VNIYz5G2iIDy+yS/00m9MPoNj1nYmxMztom4knIGReTPoGUP+AIcHZOyON6+QWztvcu4MxdlLwx7zjwHnsTvBM"
    "zw7HuHRHFk8m4ll7MrSiiXVeu3PBnb7Q23wJQwD9+zCYO3qa2+CcI78PpkMsCBiAPgCzxTX2QTYtHgfW4icPTpabPQy3zc32"
    "AdYNHgZpGOE+YObMPHLh8rDsnRv9/DhY6qjt2xP+uQ8WWgBZsU71Q+uQhqhZFIkPQxmCI2I+YoMvBQ2JmBwwTaufZPukrERN"
    "fO1r1AwFqlq3Q8GrmyCf4MnBx5Y8OE1he5L1LLVtItLLkb8Cqx0bffWJtkhGL+gpKsLAFNXHZyIMQ23SSFcLmfJJynIG2PSV"
    "1J5QnVWs0rhnSkBFTRCuvlcE+eLp91b/FziVK20KwsJVb6MOs8J2XaUtnj6NgelMQH8MEW6IX7G9wRxvAAqH9JYsWd/D4cLr"
    "vyGrxY+excyAPhBECLtXo6jyoVOL18kR8c/FWIyQI2Ijl3NNAVM39l4DM+St291hz90d794uuqONHR9+k+y8L6+UGX/NF1bJ"
    "kE7JJ8O8JcIhfNyCIeghG3Q6pf9zDZb6pK2ftoj1zwgDhIMk1VHdbNFfYdxMypfV8i7NYnb8SeI8N54o0IVY4rB+bnUKqxTt"
    "CphrAWKnvkE3HylHh0DwAKzhrwTwCDUcpMAFDJJWQ6ksoVdsBYr0qEalq2bV09KwYNFxjjRxDWoOIkLZm6SoASpO3r20xOvZ"
    "y/enJ2MU0lpfF+J7smfg3L0QwjolnU6jo2d/CL/59o/0EJpr4tPnmIe+wnlcof4UswcHjqkPh2wo7nCkMQvmXRCKl9IIiivS"
    "9kC5QiUMzUK0sgkYG1LqqCJkdCFtBfpkwGhZ5zHgipQ8e03zrMLjCyhsLyiP6+a5WGY1mfNKuYQuSFwNmmG1OHSPPJxRtERx"
    "bINJ1j5+xf4nVVVWvleUarKo+RKZoV9KeQ3r9WwGZJImaAshhJt0i9ohkKFveEYEXGmGXMMdDduG61USN6l/Qw1Jl8KmPLmb"
    "ELBSNcxDvMgLgkG7nzwg3R/ahiK3KKuErHPQ/xzz7hJ4eA1UnSb2TGgxMJcb8TsQdXZ7uVqFcQA3o268Sp4T9pwpz58c+FLh"
    "Xva8B/Vz1mW0TX0nu+xoY+MpOrWfS3lC47FpraaGrGW+BHXN1mKrrcuFDVud52VMjcM/zHeebpTezlKwEv3z7YqnNbSmGPQC"
    "A0kIcHgSmyxpFmQt3IzFMr71kc/fBEPxXWCwLFHDSjoygZq2R59z8dfCuzQ/KrXAkyfYg63zQAZ9LDOAHVYfQcb5PPDFzWV3"
    "nKDrwkMA1pGC8Qzt0hHvo97+yfT7B60Z4p7UsxAMOf9mCIfyMAh+05wZ7NwD9hNf72jKTOF1uAQ7MwFe4St5Dl9InKvvQDz4"
    "lyRNEMAgbK6lt8DXovJmcl6t5YkiHoS6h5RK3sYLyGpdGETMFyExBkKIT/8PbKGNPb+8TDZOOBATX95ZgscnQodHpP0fvqVj"
    "DgWw8yk8n7yO8zqVJ0F63HQPJiKy/SckNGwQgc2GJTQLqdTrCR6E0XLdkLcDpIc8CC2fjCEu7IXhAFT4mokF2d0RPUG5UKSY"
    "SJkjvv55KG6XOamaLX3jR4z4kOxaklzjwzNlt2fcEyGzFAy2jsTxhzdiAZSEABJgaeUag6Fps0lTy/MOM87TmOJvUljeFOWG"
    "hR6sELSGFcbFpLCtMgClTo3WfksVy0NXf40SfF2nSSiOxRyY5oIZOS0DfsJYVkPeexJ7Qx2vaK2JOvCwFKRcx1VcNKl21EA/"
    "mPu6xpCnilMAr8izGftwXWFcIYGYTVV/1Y7UYcsrxetHHlyzK0M6OBCz0SzHh8Bg4gZYcQWo89zJsx4HFIU+FE2GVu+MI6/4"
    "s0WRwCYQImv/12mRViBnI0IEUqf7JILm0cdy2nL2GGdNqzlsVoT4AYutjFYwXa8lXeaFtSgzVwx+4SN7MZYZAr1gMe5C9tqe"
    "vSLSUMcEoPn6QHQaoVBXVNTFXo+g9D3pKEAJhmuApfh5vJwm8fielQZmDi0/DMvqE/qDKiecLng23hOw+grUuRKDP8C7gc7T"
    "ZtaNlDGJKSE376Wj8A5ntRuLOxhsp8TQbdmmwb8s8/fTjxjMblMetO3FWWcvyulHgHpbhjO0M8Bcqsol20z926L2DfqhwGVS"
    "8z2ad/JBeV86RPOIbZRb6C7K791ttaP3Tvu+6QWDh3Z4cN+emSl2xm/tGaugp2s4FkuphGrI3styDSyfglfE/GOJJgpX7UmL"
    "YAsGow2GoRsNySNAoThHtj3+ayFYlaMPLM4khwPGc+H5YImgzJiToABuDtIw8C6DQMmwFIPPMmjpIwdpyay3ON0RhqapJVvH"
    "ZGLHiXaEsyUazzCMjMqfO/dyPgemr2VZRpxyg8Fv8UYFh8hSfhnnMzDiGgAho17t4NH3s0WcFS+UkJJHQT2OvhbzeJnlW2Ux"
    "4xDnIGFqNM8VTP0gwgmvmqhC2SKnd4UoCHmRV4AslILICt+fg+DEtY7J1iqXqbVAzuSgfCIicMB9kW9lqEA6FEbzdTEj6kMO"
    "fr0FvX5F0plxt9RoJjNUqiL5ViRZzaFCll4yMntQ90ecZyXGMDC6iylCGCusQw7Y682B1ZDEZ9wVSYoEz2Mp0a/ySsgETxMZ"
    "W5lK7SVGUlhRxhAukxSFcq5WuwRLBqPGpDW0bOc+PY8cOAitV5STQw9/ZSUd1gVG0Y636itxFNJ0k3K25owKVE3A4JeYrtkS"
    "gRZxNc2AeKvtCIzcWMwAIWmtbRXy8Unx3HyAwSgh4zWadkSWNctp85N6OuiTz1YzwHNxLRtbctmRyXr9Q5pHWyDfI4w7grjD"
    "+MkeAp0bF5dks8YHSUxHPAjB8FiCbOyy6F5JLpF/gUbqTYAeaWX4dmXfZ5i+tvd6L7fes2hJAs+YBF7Btr7EXRWcoYMsAFcv"
    "TyZyVCQe+H+NXrUcNFUe0VluguSlNyREWiFSCQk06qQ4Dq5sj3p3fPrjm/PT49P/Hr3++f3xefTq+PwYicc8Pzs/ffPuJ/6h"
    "hQ1MSjN0kTT3amjYWFIFMnoUMHrOfhCCieBDk+CR6ts0L6d4/iwIKFkp1h+tcNMB1v2khSDAhF75/88S137CoiZOcGTdKPmY"
    "U8JJRAKGVQDQPvw9Bt6fKVhMbYksTRoee3ilW5dhqmA3ClKtDqD8tzPGrteYmZWM90pKkAQ0Xj0ZKhZckJcWyOyn0zevzoS9"
    "gslQNkAH4WsY9KkRj9wgRGvTTb96+afjN++cbLRPoP2SiYZSq8oSdJsCz+eMomUr8C2bcKBEQiFHjueOgAfFC0Igg2zlay1X"
    "dbdcfnKP5A8XYxm4dYhLyhrYpzBtMHBzwll35yggYBtOzg28ssRzf3IektXgapf4o5zMfvqRM8LD6DrANGfwvX37h6vmDfSC"
    "PUafYD+Z2lDsYe/pvf3UPuth7mutacFqHbimbJqzS69s8JRX6MpvnbtPmASU5rTDZHiRo6y1sRYn+7TXTPh0cXg5aONXHksZ"
    "+JGS3hzMHifMUGzioplwRkRSlSsrh6lmp107hql0ddqr10pxS9JZHlfsK5FnFQND6MJ8/Zd3Z8hFZzfyVL9Hzehqk2LuaT25"
    "Eqt8XYur709TfsQ6YQNsbCL4yeTFFblFFiWomRUqXSrzhPU+OF44JjqutX8DPTIhjALLu5LpK9jGmfVzVpFkXqeEhok3qHfu"
    "17+rFHR2zLwDVX8Og6O7HrPh0EfDWYMYy5H0rhAj1/5So1dmx+G6UBeViTLYX2uvmI9Og8JcQ3GWwhmEz5jXI9kQkzr6x8HW"
    "XsglHjfl8kOc4TzVTqqZAX9p1nEurkHWo0JeZ8kaV4FZ20ojZwWXcrZYGa838XJV28lcGK7hpgCLbTYZ2dbpJErrZb/ZUfqt"
    "XB3IZehVl4XaQ9w0YrvouWtKnSnIGAVhE4r/AejBBGOtRa9X8rYDJvLzoleLbZ0BukSxXk7R1SZVnV4t+yHOR+cHqZADMXtU"
    "b9lg8Fg+qbg2kmRXa+1jIPsZY4uvEMiJkJxFZqzcFi0/hIl9I1FSDEX1wAMCQHEGvRNxjo0XcGAlxRQhAkVAcBJaNCEEqYJx"
    "dBFamTlTT/Jec3cOL4J6dDGSAktyB4tfevKRFxj4ypvC6YWkDBJH5XPhMcprwISzi9Jn4bJJX4JvraDfaVltED8wMcaO4lzt"
    "bWGjZQODVxtrZ5C1gajRD1XnNvfHscn+ghE3X8LpiJgIAQcRj+jbdM52fYRzY4lB4R38H8yUlc7NA97BB6YGK2pLGBerxtZ8"
    "fPbtHldwX0ouyu0S+FFW4O4/AghwQsaHC2QJVkg+zQpMjnRSfsf/xB3AqN4/gnwVSYXxh+pwKX0BL5GtASUth5KtL7ARTmrP"
    "kIC01ALNqJTQiCnNdZWnt6MlTKIe1Sy9UHjL8RIh73i4rq+42S958QZXKF6V8sqWYvKwOpRCtZaLynEzQw8Mu2VUVCkHkVFx"
    "/1rKOs4QiUEPwQVwKvMiW2Iqi8RdrFUO1O7TuieqxNa0FZSSspkNHnIsgQKZx+SJiSvpMRPxJt5KvxH5umgPYtap0DLnk4NW"
    "RZHVCymrWeKhuU7ilOgmCUE3IwcbdfE5tYA2UueTgGUf4Sd+vAtc4fj5Ug81jTev9rfNkpCbsP9++7dfe9uCyIYtmYXYgG4Q"
    "RUm5BgsqaqTMLOalssBXyTTC71JXBiRjtCoj/pGR0MLLZv4Rk2zIWJCKjh9gJqtzjDCFFqHTTQv4ENJ2+VmAmYj0+dKOqcJw"
    "7fSKft82sTQ2E8Qd/fldtSMVpyi13sW6MqZP41aiUjWjKxnqgGC+tdcCqc9oKF6SZaQuI447Te9k9smdu7DPwNQu0I78oizo"
    "Lo8U28TH0A0in0t+liaRlnAozluinHrJr0oh0gnocoNn6FaWu4vtfdUgkF4t6VxPVU4S+q5srwACkDqyOHkbr5ik6N6mxL89"
    "xFcU3M1RSXVvWnLG46YU7M1FNV2xMAz+YPADr4/C8WVdKLIWclc3Yykz1dOQ+GKGWJGoNg8u6uayLx0Dd6qmVB+JZencjWEa"
    "TR++yvkcveQTOmr+0eEhSI2hcP+aNBQiApeiKU+GwIeKErLAkdEfHcpB13qBwrqWFONKoTjDtChmAf7HocjcVfJAKA1hvj40"
    "HspH8iuC5CVZ5EAtpNMhUpiwEM4QHsL2XtwqFjnkNDlyqZP0HfTuiOUf1b45SeA8ztegv2rSwEdipObtbjtwmHha+7DoF2Au"
    "jb5zMammdeEoDKgbsKpgVAe/bshPmFgXDXAhT9QjlaAlyRN+A/VCwTcqQZ4Did8T+3rHTFvGvAgzqIjPM5JTCec6jJU6wLYu"
    "GfpLN2hmi8xZn3EMkGfsVwAhCRvRuQQ2y1PyPKBpyKqcfSHU3DxjB7fKfQO1vwQVDgUtmPAtO1EHUzqKXMe3a8XE9jh3exXB"
    "Rzh1P9eh6+h7j3HoFhbtUDpRWvgwL+XRw82Bpz1RUOZCSUJZj5iH98nBi9xXiQzOdlQ5i+tmpzCs0iq5ubGPnaxIq6NZAiKv"
    "lWupcEjb0X/6t0ogkScC6AvhtnM6nUGQN0nvkM9MQTNQmkIkMYiDSgTRVBzEdpszQjnQRu3p8g6iM9jjaDea+bJMUt8cxXgN"
    "muvfpXL7d604/B355pA959qRDGs8OX19/PIkVGS+vM/vrJuT4QXjGDdlmJcbsHud2S4pzUGab9gaLSWVyCPnhR9hYmDBk73O"
    "7dQaAQFRO27bNUHqmWOGaFsmYls7L6+He+9TC7rxCgffTbHT0Xy8S1evUdDPQBOrqmx/kF0ZMdIQyeZ25lpGSQ2d2xmdy56k"
    "Ztj8CfN7o79EX1tWpeJXoTiWSrq0nQiGiHOEvcXjnq+TVF17d9moTHSzuR8wTD7lS0zmTjiZ+1DrPWpSaIVQenpRjsqV9vtJ"
    "HxsvDTYKF1drB6J2xZFjJuPbhXGLw3JAb965AcvKskwXRmJHDtQmf53xTL9PmKjQS+LrJ0x6QScOgfQjfxzY4ULKEcflawhE"
    "nPf2H2jpagQoTvc32dOs4cwuYDbIEgisw62GgoSgGqgra6Az4HOuUBiR0bfzjDLwOL5Xz2y21wfRYoOotcgzZaUxl9fQ7wJt"
    "6Mv9tvx1/AkTHUF+pNsdkSgcpI30GxCJtw0aSlmRd+bHlrvMGU8ov4Mgv4Ow/A7KLSoO7myusTsYdobqaCLsot8/6B3ycLU3"
    "wQ63aUS4poIU2XTNSRfWeetYdgbPX3sOayWaU1hR/BKL5Jg84s6NtVY2MXHGyQq0xsZwPTBd3UjslbxzdiXQpOXSMnIOF77M"
    "WidFZ4jXWi61HNHz4MP6cIazk0Ecr7KhwHRh7L03bbh9v1Ha//YutNLFPmWxuAPgO4lNtsYn+8LW/QNI57FNL/BVRzPxPgaB"
    "RVR1+JQrDZUQvO/2RFem4YQfEdDrn30vsZrwEgavKMxCQmQsbH/Fbthz82Du7T1D3mfmFDpTmrUyAd0omJpmK7eQLvJ006NI"
    "GTP5kJ2rrD2EQ7+KO2p6oJseXGIGqhml7SLZWQE0gyr0u5guICmZzmpsr2LZ3VsVPVeYO5cnZKBpU5ur5l1fTvuCtWuU4w3O"
    "iXWNtYMcsNGHreu6w9aVW7PPEvXWcm1nRC9944pBf6S0bMuW+G1Uw0Ww6LKKeH385ueTV4ZC7g2QlLxc63oq5Yb27wLi7Im6"
    "kOqsPUzWyxU280v7SisLUVYEula0bfAb1kFsZm96cH8oAaRCXdE4v011vkdP7v/XVbWNQu1nqDQdPTJ/vC/TnQH05yk9xL/m"
    "MRz1pLv7eFpU2rISYIGzBX2jOqPxjbm6XFeztEZ+MLs46LH7gFmQpuDXAZtdfczT8hUMXUi2SYiglnQ5UyU+0a8jnkkfXGlK"
    "urgvIlgenM710j9q34R0b0GC5uFx3LSlsXp0Y/FRooS1JMqkBiTh2Lt9yhIfWOca6p28vnZgPTxQYceDIi4OAr5GKPY7xLVV"
    "pYF1fuoF6epaSDHKIuU6XqDBbKFTYvSm90XaKg82top92cndVJ0pwdiSrhIoWXNo1z7p19zwiA3FAsuJyCn0SID/e6rXPyR+"
    "P1v49Ug7iZ3/zyQc6eiPkXFIy+t6YeFnfwUMV1h9psy7j532ijirR2s1vI4EUNgz/e4JdGycCO04GCzvM3Y+ltP6MSbP2YrC"
    "R6q2BvlrrrDzlan0F4pXfD09JnNTXwKQbhSOU9fxPA1baUVLzJuSYKgGXi2WK/a0oqrhamb0HEeGn/AaM+huSwzQAbrwKZhY"
    "RaBtbG7nCCqFrceagRZTuZ7Af1KfTJNrugnZcCDlAjb/SAVC0JMpnj6l0Y1PmJ7+R/OwFRWiyZtpgmZPqUkI7okASYSrEd9L"
    "UOQGPHR1IuzQun6Ec9R0R1NVM34iBxh1VA/5+4QbdHX+u2KnyhDIu5RkwtNYoKq7/N2HZ3ju8EclMBCaApY+tuyj8FVRR6zo"
    "iN0CZc7MmlskhZXKLm8ARb5Xr+KNyuOiu0/QLPxQAoM2s6Xr0yt4ZqF9sS5u6AYoPKbUcynNhmDJPyxygkfphrjzkgmj1x/n"
    "InmnklEVVzDAyWATOSvKoNLfL3WPEAPWmE4/4QtuwFzH8N/F4WUfg/jSBVlUGUvh26UwqWQJOua//F1wHDHCEf0GODOIQkrD"
    "bTmlX2KlTvbS5OvlFNkKul2ZKeElo6G5wIH11rCBXfUZYfyaVtk8U0WMWaHEDEu7Dl1tld0lxicLuAJbV3dGsLd28rJc4rJF"
    "wL9M9mlf/SEOqFPx2bloV6lkHldLJkqZ8+xnppIxfH7sAjPsTHcqHquSJJxP1JPw2aTLFdXaId0D666h2OHFMO7R5So/kZeU"
    "24fLmwQ/+6q6U76aR3jkI9w83LtInk0l6AwZoSdFgrRqKSzR54Rl1agghSooTJUGrSqHR4e9tQ51VePe61XdkofCU4VQCZxT"
    "+lBVmpKSSPIEYtXd4oZ61sqTBkRJB3xgEuSVTGXC8bquCJRfh1aGve85BZNb/dr13Cb28h0o2n3ZgmDXQptYqHA6s+un1dMp"
    "fDaxcOV0VXnjdt3JnopxE6tMobt6u7IyV1BUl0Y93gqHRC5l50ulfYMl3k9Izj7r+tR7KIp+702B3FdaU9Ws7i+zqX4nouuF"
    "6tK5JL9nuOJn/dRn1uqQ3xOX/rjctimtbfb0Wc+OqDU4O8Ig2iTxrEsQBqk9/S3B6slV9W6koiJkpKAXQ1tjthiSA30s0ERI"
    "xaO8/v6CmIYF4mn8dPrUhnPoAILmDkWpDNyhuMaa7CSeEc/jjjIvxN0BMFFxgOycGpMed4D2ycEONCt0CFvWP7T9/YQaqpuJ"
    "LvfmK92pqBfxCkuO0bUCLMqtLhXIG9Xk6pluSVJxUeVHuUT/aEcBbQumH90ZqN53bhVQO9VgLEbfYR4VqJxZbwVfaPCf4ecn"
    "4jA8El9DIxdWb2rzWByFR9/1JTiPMdbWAtGX3MzthEcB5D1jHIbftCBx6/4x5a/7B3NBzeOoSle4kmeH4bfmx12gQpvEzjGW"
    "0q5O5xikUmQOHa7JFdscQYt3+jfAdch5RKB3jij25IV/T84PNoTvNtDVBqo/JPsFIVIXlxoKLsbfUAGAfZfmCrCiKbu5WFmJ"
    "oJjif5Hr9AiqQecHjxnuaHxpChTGFdlcqxA+xFsf4JpDxOXu4CjQLW0szYqK8ZD+AhAnQQmFIYWeHiw3jdWrjb7WTmGnsz1R"
    "4OQMer0VD7AEWYbyDv7uVMJqy11pMYnW8GjokG1WBcZ0ANzsiJ0dfS57InaBGq5VGa/jVZ17Pu/0pl2johC0OaLEvIYNWHlB"
    "P6d7lMPHTBknJ4mL6haNkJS6Dm3SXcm9wSQK+uieHJ7Tkw/vPe1AwHaog6ujltVwxtyDB03w8uRq7txrrLeqRwFyvvGBL+zt"
    "dc9le/RkY9VHdYwshd6lgp5SYy22YHTAPh+hi1niEt2y+G2bYtyiRVOiTJ3ZdoEy42SX0gfpc76wj3YAEuLZnsPCDFFLQHyL"
    "QI1ecwSm3BLPeyu/zT2nViYX1DQGGb0RpilLlU+RIcGl1SyrO/kUCl58jUHJhqWuLEcrL3EC2afTEmiODBo7yRotLe83ejZB"
    "azHxXkkPnTqikvS1d8SwfW1hQZODD8dnZ3TOYZL2KXed+uXNl/cXqNfgyKJTWDbzn1XIVQ4l6+XxezAi2FZ5bZfZd5/zgGu4"
    "UiJ7VmytNyaZF1jEM7wQmm/Vq49Uyhq2lTcrUWpnhX4hEOyTflsQ5U6ZqhJMc1QReIq3FyQT1KOSSS2OQdFLsQqKxOAqk29B"
    "IvEE0qFBNxVft1nGYNhXVBGHoRnHTJ5xERy8ccrXWvGFWew24EAq0OGH4/M/2ZVonarFfNWiaDAIe0Av/zgg14NdUBmrAlHh"
    "+7cf3mgRwNydRQPdXY2x4L6Q7/14gcXrefrmLpH9ri2+c+o6KsyuUjRFf8GEc2DBBolSmaI3LZj4tSq/KH/AAqmkW0WR1IlY"
    "9eqt1qjzbgkFFPPrVG226jn0F3Wld88oJNrv+ZKbjFDGTHaMNryDidTSyYRyrcU25h+BanmJJin7Ei1a5SuZuX+hapW+99Xv"
    "6M1t9eKvRc+ivK/ar4Kw39+mX7DV9/q2ffB+kiXvqPkDr8Zx3tTVC3B+gHQmvDtDgDsPvjIY/PgffgAOfKDxOlssy0Ri5rD8"
    "T99++y+o0om+qH8Cq6UXB/mmLgJeNONP0m6n8yRZbl+lTl0hYe3cTsAYsGAva60u+bHo15f4yJO7XlHO8RbvdTDDeo3XfpG7"
    "fF5Nq7G875jG8irg3gxMnZqYoizen+rMWcEyw7deoMKzUZ5kWoAWIouYk4235ZoKgTfE5xepVBxUhjSY5kNZa4F/Aj7QxBXo"
    "rqksskV1IGSyin4bCtXq5No+KK2wpoLDQt2Xccy9H+7srcNMWFQUrEesMvwLI+luTUEdHCXauzQ0wxVD2/mLD4XaFcX2A79T"
    "P2NS2+PT2Sih7aFUNv5ynw+mA6EVtp+m15SbyTF2SsykR/b1M+5qqhDetS6EyiuO1C/Y7RMqYNndWaMgqAxVdZ4BzGoXeP0Y"
    "5PFR6wRBR0YJPwnUBvWE/rt+hDZUo9DcmZ2V79TyNejHXOehnLPOnaTHXebpkKKBMObAaD2DXYfT59fBcychiZOPxm2aka9f"
    "wJs9qp+nPEHtmz98yVelEvVdKYI+Mt+9u62ouvQmLqFc78l02rUMCn1C9DJr6+1lpXybprrXxoVkB/rysBx3/wa3c6TYl6Gf"
    "BrvANj9NohZf3jXtxvfYkvTv7maHuNM9Lm4ubdeBtojVxPbKBYoUJ6F4M+cbdFSzkM3OrrLmkdCh0llobFtHFVeBATxZgosK"
    "PpG0Ufq5qqTcAUk1YSpQg+pQ0X9ed1CsMfzufQfJXMyTtPm9y+wM68pDKnUTdhGovqrIrBKEinkfJ1TucmgimAoBpgpRZ2hO"
    "ST+oWyhUsleOwWglLbVWNjpJhQ48H7UNvGcwBYEbhI/LgO+XToZgnIoGdlW2sbnjbo4GPXlsqvsjs9z7a/J+0Ss4+46xuXil"
    "ik743Vx4vEpilTfZ43HhuyTBWOy5/ISnmNY0fkI5j6cnv3jtFa8wLY1ZRPtG0FBYKRQ3n8ZihBeLbz5dHF0GwcX4j5d7ixUC"
    "2K6Lq5fTjJ99U+/E3afxk++sHMq9rEZV9yBOqpJ2OeQDBlsBBh2+EJm1XaMm9mEPY2z4mhMqATs0x4WqsIKxWpMtJS+o0YUh"
    "UoT5xZn0vuVeqBtVljX8rV4ttdRG19mUJEO+oXvz0/XrNEMCBjJ65nfcXgo+sjV8iQjqIUOliEhKwOTmXfvukMylQ7OJ3hBn"
    "O4pk/JhLOmxBt8L3vR2NSVGm31QNJ1KUOQKr3QXWq19J3uLPnfy0Q2xqMl3kzc8jCwitqwdAvOIX0PH7Ao+ra6pjy95BHzpd"
    "T7zHvAvWwnW8CjFTPpagfD04lpetruvJM1Dh0yb+FFcT3wO+huGRD69+pAAUvxAlabmae0CaSLxMEJh03yto/8OKqxMPyeYH"
    "JyNAM9gaHVbyJctCv2T5/nVJ2wQmwbc4J3BuqEJA5bzpD1dNFR+4bBmCqH0TaO/l22woY8uQPsoolP56pL7K1chvjrVkna43"
    "5AAmL1LfwfKoYPdYOC/INY5jciuqMiqW/7A39YAMeCTzukkAbND3qpk/PLLWbv+Z7QH4TedEWLQv9fwe6lcHfy+Jizvr1bvO"
    "nYfuMI7n58sftftJ0R3cnDAPfbPe55ws9kpZB8v1jH4WWa90IFp514lKnUeKrGnYbk6zcm3K7DM0q7v69vxgRbWViTy1D31s"
    "XOhjdLftvIP9G4jUTswYyykjgz685OjuaEEpOCPkIPSJPgRtIoqipJwpR7AzAD2RB/O+UgDyHeNchFZVZPDplaiUqDZHN5hc"
    "ukxpxoH3wsOXX1P6D187OVLvGWrnpfanA5mNJN0lxXLW9GpbqiVrZyJ1spPwdRhWMo+bLBW4rxukCAKmc+GN7H6ZL3nTHQ6/"
    "41pcTMuY3f4IZvPsX7QKHfQ1VzJwUcFjVkWyCKGouz26++7Rqxz0XAV4JGOjNMqSCkzONolxiLSgyZyyibzDKb/iTSqkx8kd"
    "/n/XclEofKpO6jv2IuxOem6Een0+HTjWBaXL+/03FnQK/wtxxAqQ0869YCHBAMCHLzoooWoP0C0MLgfcA7ENy3E49Urnf0gy"
    "94QywLRYZSu62Y/arcbLSD4B1P7+9zIoJkazrrV70H57QqR7Pu97GMpPloc3OOidF0X3qlS9WcgJaVFoqCXfZAhUvUGq6c61"
    "Fehk4x4Lk2VNeN+BcvST/brJ/XqJ1kn2v6/R3VyOV3BuAV+yfK5f1IjFemVm0UOz/nagi621b+tJq3CzKDFnE/GGocO44Cjs"
    "uhjLNIaNyj7/qj8SIguRkNdKvWFSJRAmgqI2XBuFMpEoCCnBsX9ratLIfecNiFkTiCnIcFgjLJ93Xfq8OPKYl+UNvqD0JtXZ"
    "jbKupJ0xEes3O8JcQnEWb5FW8nKd5FtV1CJ0RH6x9XsytuQlzs47MDtiH21GZo5/Pj599+bdT2ODF3bmUVWtFvy+46mLXJxb"
    "CGzhjAvIUPCHw1UdKL27xsFgTBbQr2S5bwK/1Fi9mop7jV2vmyztt8/ZNxR9Dr9e7+R947MHL9ZvMTC5BubVBQd1aN3awBfV"
    "dYyRznEZ6IQgeRdhz2vCOVDdFUBcr+z+xEoNvidL3cmw/AeZyx/75DPn7+EQu6ddGS3vYY1egKhfNy0PBqipAzgSEV07jyJK"
    "YY4iupEQyfo8bAdkDd1TwCpf/wcGuVCq"
)

for _name, _blob in (("lpf_nb.py", _LPF_NB_B64),
                     ("pyrosetta_rosetta_scripts.py", _SHIM_B64)):
    with open(_name, "wb") as _fh:
        _fh.write(zlib.decompress(base64.b64decode(_blob)))
sys.path.insert(0, ".")
import lpf_nb as L
importlib.reload(L)
print(f"lpf_nb.py written ({len(open('lpf_nb.py').readlines())} lines) "
      f"and imported")
print(f"pyrosetta_rosetta_scripts.py written "
      f"({len(open('pyrosetta_rosetta_scripts.py').readlines())} lines)")
print(f"{len(L.__all__)} helper functions available as L.*")

---
## 1 · Get LigandPathFinder

Clones the upstream repository if it is not already present. The notebook runs
`lpf/lpf_coarse_path.py` and `lpf/lpf_trajectory.py` from it as subprocesses,
and copies the stock RosettaScripts protocols out of `rosettaXML_files/`.

In [ ]:
#@title LigandPathFinder location {run: "auto"}
# existing checkout, or where to clone it
LPF_REPO = "/content/LigandPathFinder"  #@param {type:"string"}
FORCE_RECLONE = False  #@param {type:"boolean"}

LPF_REPO = L.find_lpf_repo(LPF_REPO) or L.clone_lpf_repo(LPF_REPO, force=FORCE_RECLONE)
if FORCE_RECLONE:
    LPF_REPO = L.clone_lpf_repo(LPF_REPO, force=True)
XML_SRC = os.path.join(LPF_REPO, "rosettaXML_files")
print("repo       :", LPF_REPO)
print("protocols  :", XML_SRC)
print("demo inputs:", os.path.join(LPF_REPO, "demo"))
print("\ndocs worth reading:")
print("  ", os.path.join(LPF_REPO, "docs/config_reference.md"), " every run.json key")
print("  ", os.path.join(LPF_REPO, "demo/tutorial.md"), "       the worked walkthrough")

---
## 2 · Rosetta

LPF calls `rosetta_scripts` as a subprocess once per path node / trajectory
window — it never imports Rosetta. So a plain `pip install pyrosetta` gives it
nothing to call: **PyRosetta ships no `rosetta_scripts` executable.**

`pyrosetta_rosetta_scripts.py` (written alongside this notebook) closes that
gap. It presents the slice of the `rosetta_scripts` command line LPF uses, runs
the RosettaScripts XML through PyRosetta, and writes decoys plus a `score.sc`
in the format LPF's `ScoreParser` reads. LPF itself stays untouched — it just
execs a launcher instead of a compiled binary.

Rosetta is licensed separately from LPF, PyRosetta included. You need your own
academic or non-commercial licence; installing PyRosetta is accepting it.

### Two ways to get Rosetta here

| backend | how | cost |
|---|---|---|
| **`pyrosetta`** (default on Colab) | `pip install pyrosetta-installer`, then it downloads the wheel | ~5-15 min and ~2.5 GB of runtime disk, **every session** |
| **`binary`** | a Linux `rosetta_scripts` build you keep on Drive, plus `main/database/` | one-time set-up, then instant |

PyRosetta is much the easier start: nothing to download by hand, no `chmod`, no
database path, no architecture mismatch. The price is paying the install on
every fresh runtime, and no MPI.

If you already keep a binary on Drive, that stays the faster option. Nothing
else in the notebook changes between the two.

In [ ]:
#@title Rosetta backend {run: "auto"}
# 'auto' prefers a compiled binary and falls back to an importable PyRosetta
ROSETTA_BACKEND = "pyrosetta"  #@param ["auto", "pyrosetta", "binary"]
# pip-install PyRosetta if it is needed and absent (~1.5 GB download)
INSTALL_PYROSETTA = True  #@param {type:"boolean"}
# interpreter that has PyRosetta; empty = this kernel
PYROSETTA_PYTHON = ""  #@param {type:"string"}
# PyRosetta backend: worker processes per Rosetta call (~1-2 GB RAM each)
SHIM_JOBS = 1  #@param {type:"integer"}
# binary backend: full path; empty = search common locations
ROSETTA_SCRIPTS = "/content/drive/MyDrive/rosetta/rosetta_scripts.static.linuxgccrelease"  #@param {type:"string"}
# binary backend: empty = find it next to the binary
ROSETTA_DATABASE = ""  #@param {type:"string"}
BACKEND_DIR = os.path.abspath("pyrosetta_backend")

_info = L.resolve_rosetta_backend(
    backend=ROSETTA_BACKEND,
    rosetta_hint=ROSETTA_SCRIPTS or None,
    python_exe=PYROSETTA_PYTHON or None,
    shim_dir=BACKEND_DIR,
    install_if_missing=INSTALL_PYROSETTA,
    shim_jobs=SHIM_JOBS,
    database=ROSETTA_DATABASE or None,
)
ROSETTA_SCRIPTS = _info["rosetta"]
ROSETTA_DATABASE = _info["database"] or ""
IS_MPI = _info["is_mpi"]
BACKEND = _info["backend"]
print()
print("rosetta  :", ROSETTA_SCRIPTS)
print("backend  :", BACKEND)
print("mpi      :", IS_MPI, "(n_proc > 1 needs an .mpi. binary; the shim cannot)")
print("database :", ROSETTA_DATABASE or "(PyRosetta's own, or found by Rosetta itself)")

---
## 3 · Project settings

One directory holds the prepared inputs (`files/`, `ligand/`, `start.pdb`,
`path`) and one subdirectory per run. Paths written into `run.json` are
absolute, so a run behaves the same however it is launched.

In [ ]:
#@title Project {run: "auto"}
# all inputs and runs live here
PROJECT_DIR = "/content/drive/MyDrive/lpf_runs/peptide_demo"  #@param {type:"string"}
# chain the peptide gets in start.pdb; must be free
LIGAND_CHAIN = "X"  #@param {type:"string"}
# 3-character PDB residue name
LIGAND_RESNAME = "pep"  #@param {type:"string"}
# MPI ranks; >1 needs an .mpi. build and n_proc <= nstruct
N_PROC = 1  #@param {type:"integer"}
# zip each finished node/window directory
ARCHIVE_WINDOWS = True  #@param {type:"boolean"}

PROJECT_DIR = os.path.abspath(os.path.expanduser(PROJECT_DIR))
FILES_DIR   = os.path.join(PROJECT_DIR, "files")     # protocols + options + VRT1.params
LIGAND_DIR  = os.path.join(PROJECT_DIR, "ligand")    # lig.params + lig_conformers.pdb
for d in (PROJECT_DIR, FILES_DIR, LIGAND_DIR):
    os.makedirs(d, exist_ok=True)

if N_PROC > 1 and not IS_MPI:
    extra = ("The PyRosetta backend has no MPI: set N_PROC=1 and raise SHIM_JOBS in "
             "step 2 instead (L.set_shim_jobs)." if BACKEND == "pyrosetta-shim"
             else "Use the .mpi. build, or set N_PROC=1.")
    raise ValueError(f"N_PROC={N_PROC} needs an '.mpi.' Rosetta build; you have "
                     f"{os.path.basename(ROSETTA_SCRIPTS)}. {extra}")

print("project:", PROJECT_DIR)
print("files  :", FILES_DIR)
print("ligand :", LIGAND_DIR)

if PROJECT_DIR.startswith("/content/") and not PROJECT_DIR.startswith("/content/drive"):
    print("\n[warn] PROJECT_DIR is on the ephemeral Colab disk. Everything here is lost "
          "when the session ends — prefer a path under /content/drive/MyDrive/.")

---
## 4 · Target protein

The peptide is pulled out of the same complex in the next step, so give the bound complex here and keep only the receptor chains.

Anything non-canonical you keep (cofactor, metal, modified residue) needs its own
Rosetta `.params` file, listed in `extra_files` **and** in the options file's
`-extra_res_fa`. The preflight check later verifies both.

LPF starts from a pose already relaxed into the Rosetta force field. If yours is
not, minimise it first (see `demo/tutorial.md` 1.3) — an unrelaxed start pose
shows up as a huge window-0 energy.

The default is a worked example: **1CKA**, the c-Crk N-terminal SH3 domain with the proline-rich peptide PPPALPPKK in chain B. It is a real structure with a real defect — its C-terminal carboxyl is truncated — which the next step detects and repairs.

In [ ]:
#@title Target protein {run: "auto"}
# a local PDB path, or a 4-character PDB ID to download
PROTEIN_SOURCE = "1CKA"  #@param {type:"string"}
# chain of the bound peptide in PROTEIN_SOURCE
PEPTIDE_CHAIN_IN_SOURCE = "B"  #@param {type:"string"}
# comma-separated protein chains to keep
KEEP_CHAINS = "A"  #@param {type:"string"}
REMOVE_WATERS = True  #@param {type:"boolean"}
# cofactor/metal resnames to keep, comma-separated (e.g. HEM,ZN)
KEEP_HETERO = ""  #@param {type:"string"}
# True lets Rosetta rebuild hydrogens
STRIP_HYDROGENS = False  #@param {type:"boolean"}

src = PROTEIN_SOURCE.strip()
if len(src) == 4 and not os.path.exists(src):
    src = L.fetch_pdb(src, PROJECT_DIR)
if not os.path.isfile(src):
    raise FileNotFoundError(f"PROTEIN_SOURCE not found: {PROTEIN_SOURCE!r}")

print()
L.pdb_summary(src, f"{os.path.basename(src)} as downloaded/supplied:")
print()

keep = {c.strip() for c in KEEP_CHAINS.split(",") if c.strip()}
het  = {h.strip().upper() for h in KEEP_HETERO.split(",") if h.strip()}
PROTEIN_PDB = L.clean_protein(
    src, os.path.join(PROJECT_DIR, "protein.pdb"),
    keep_chains=keep or None, remove_waters=REMOVE_WATERS,
    keep_hetero=het, strip_hydrogens=STRIP_HYDROGENS,
)
if LIGAND_CHAIN in keep:
    raise ValueError(f"LIGAND_CHAIN={LIGAND_CHAIN!r} is also a kept protein chain. "
                     f"Pick a free chain letter.")
print()
L.pdb_summary(PROTEIN_PDB, "cleaned protein:")
SOURCE_COMPLEX = src

---
## 5 · Peptide → one Rosetta ligand residue

Four ways in; `from_complex` pulls the peptide straight out of the complex you
loaded above, which is the normal case.

A peptide taken out of a crystal structure is usually *not* chemically complete.
That does not matter while it is "chain B residues 1–9"; it matters completely
once it has to become a single residue type with its own partial charges. So
this step:

1. extracts the chain and reports every missing heavy atom per residue,
2. rebuilds a truncated terminal carboxyl (planar `O`/`OXT`, 1.25 Å, 120°) —
   but refuses to invent missing side chains, which you must fix upstream,
3. transfers bond orders from a sequence template,
4. sets pH-7 protonation (`-COO⁻`, Lys `-NH₃⁺`, Arg guanidinium, charged
   N-terminus) — the neutral template would carry the wrong charges across
   every interface the peptide crosses, which is exactly what the energy
   profile measures,
5. adds hydrogens at the bound geometry.

Histidine is left as the template made it: the tautomer is a judgement call and
Rosetta cannot be told which one to use for a ligand residue.

In [ ]:
#@title Peptide input {run: "auto"}
# where the peptide comes from
PEPTIDE_INPUT_MODE = "from_complex"  #@param ["from_complex", "pdb", "sdf", "existing_params"]
# mode 'pdb': peptide-only PDB, in the receptor's frame
PEPTIDE_PDB = ""  #@param {type:"string"}
# mode 'sdf': SD file, first entry = bound pose
PEPTIDE_SDF = ""  #@param {type:"string"}
# mode 'existing_params': dir with lig.params + lig_conformers.pdb
EXISTING_LIGAND_DIR = ""  #@param {type:"string"}
# mode 'existing_params': PDB of the peptide in its bound pose, atom names matching lig.params
EXISTING_LIGAND_POSE = ""  #@param {type:"string"}
# override the sequence read from the PDB (one-letter)
PEPTIDE_SEQUENCE = ""  #@param {type:"string"}
# build a missing terminal O / OXT
REPAIR_C_TERMINUS = True  #@param {type:"boolean"}
# COO-, Lys-NH3+, Arg+ instead of the neutral template
PHYSIOLOGICAL_PROTONATION = True  #@param {type:"boolean"}
# empty = find it in your Rosetta tree, else download it
MOLFILE_TO_PARAMS = ""  #@param {type:"string"}

import shutil
LIG_PARAMS = os.path.join(LIGAND_DIR, "lig.params")
LIG_CONFS  = os.path.join(LIGAND_DIR, "lig_conformers.pdb")
LIG_POSE_PDB, PEPTIDE_CLEAN_PDB, PEP_MOL = None, None, None

if PEPTIDE_INPUT_MODE == "existing_params":
    src = os.path.abspath(os.path.expanduser(EXISTING_LIGAND_DIR))
    for f in ("lig.params", "lig_conformers.pdb"):
        shutil.copy(os.path.join(src, f), LIGAND_DIR)
    LIG_POSE_PDB = (os.path.abspath(os.path.expanduser(EXISTING_LIGAND_POSE))
                    if EXISTING_LIGAND_POSE else None)
    print("using your existing ligand files; skipping peptide preparation")

else:
    if PEPTIDE_INPUT_MODE == "from_complex":
        PEPTIDE_CLEAN_PDB = L.extract_chain(
            SOURCE_COMPLEX, PEPTIDE_CHAIN_IN_SOURCE,
            os.path.join(PROJECT_DIR, "peptide_raw.pdb"))
    elif PEPTIDE_INPUT_MODE == "pdb":
        PEPTIDE_CLEAN_PDB = os.path.abspath(os.path.expanduser(PEPTIDE_PDB))
        if not os.path.isfile(PEPTIDE_CLEAN_PDB):
            raise FileNotFoundError(PEPTIDE_CLEAN_PDB)

    if PEPTIDE_INPUT_MODE in ("from_complex", "pdb"):
        print()
        rep = L.peptide_completeness_report(PEPTIDE_CLEAN_PDB)
        if (rep["needs_oxt"] or rep["needs_o"]) and REPAIR_C_TERMINUS:
            print()
            PEPTIDE_CLEAN_PDB = L.complete_c_terminus(
                PEPTIDE_CLEAN_PDB, os.path.join(PROJECT_DIR, "peptide.pdb"))
            rep = L.peptide_completeness_report(PEPTIDE_CLEAN_PDB, verbose=False)
        if not rep["ok"]:
            raise ValueError(
                "The peptide is not chemically complete (see above). Rebuild the "
                "missing atoms before continuing — PyMOL, pdbfixer, or a Rosetta "
                "FastRelax with the side chain repacked. Converting an incomplete "
                "peptide would silently give it the wrong chemistry.")
        print()
        PEP_MOL = L.peptide_mol_from_pdb(
            PEPTIDE_CLEAN_PDB,
            sequence=PEPTIDE_SEQUENCE or None,
            physiological_protonation=PHYSIOLOGICAL_PROTONATION)
    else:
        PEP_MOL = L.mol_from_sdf(PEPTIDE_SDF)

### The conformer library is the peptide's only flexibility

`-packing:ignore_ligand_chi true` means Rosetta will not sample the peptide's
torsions during docking; the library below *is* its conformational freedom.

LPF filters the library to conformers within `conformer_rmsd_threshold` of the
**current** pose at every window. So the library has to be a *connected ladder*:
a purely local ensemble would never let the peptide unfold on the way out, and a
purely random one leaves gaps the filter cannot cross. Three populations are
mixed for that reason — local perturbations, medium perturbations, and free
embeddings.

The report that follows tells you whether the ladder actually connects. If it
says `NO`, raise `N_MEDIUM`, raise `SIGMA_MEDIUM_DEG`, or lower
`CONFORMER_PRUNE_RMSD` — a disconnected library makes the far conformations
unreachable however many entries the file has.

In [ ]:
#@title Conformer library {run: "auto"}
# small perturbations of a few torsions
N_LOCAL = 120  #@param {type:"integer"}
# larger perturbations of more torsions
N_MEDIUM = 120  #@param {type:"integer"}
# free ETKDG embeddings
N_GLOBAL = 60  #@param {type:"integer"}
SIGMA_LOCAL_DEG = 12.0  #@param {type:"number"}
SIGMA_MEDIUM_DEG = 45.0  #@param {type:"number"}
# minimum RMSD between kept conformers
CONFORMER_PRUNE_RMSD = 0.6  #@param {type:"number"}
# short MMFF relaxation to relieve clashes
MINIMIZE_CONFORMERS = True  #@param {type:"boolean"}
RANDOM_SEED = 2026  #@param {type:"integer"}

if PEP_MOL is not None and PEPTIDE_INPUT_MODE != "existing_params":
    PEP_LIB = L.generate_peptide_conformers(
        PEP_MOL, n_local=N_LOCAL, n_medium=N_MEDIUM, n_global=N_GLOBAL,
        sigma_local_deg=SIGMA_LOCAL_DEG, sigma_medium_deg=SIGMA_MEDIUM_DEG,
        prune_rmsd=CONFORMER_PRUNE_RMSD, minimize=MINIMIZE_CONFORMERS,
        seed=RANDOM_SEED)
    print()
    NBR_SD_INDEX = L.suggest_nbr_atom_index(PEP_LIB)
    print()
    _D, _rep = L.conformer_library_report(PEP_LIB, nbr_idx=NBR_SD_INDEX - 1, plot=True)

In [ ]:
if PEP_MOL is not None and PEPTIDE_INPUT_MODE != "existing_params":
    mftp = L.ensure_molfile_to_params(
        os.path.join(PROJECT_DIR, "_molfile_to_params"),
        MOLFILE_TO_PARAMS, rosetta_bin=ROSETTA_SCRIPTS)
    sdf = L.write_conformer_sdf(PEP_LIB, os.path.join(PROJECT_DIR, "peptide_confs.sdf"))
    print()
    # nbr_atom_index sets NBR_ATOM directly, so molfile_to_params also computes a
    # matching NBR_RADIUS. Rosetta overlays every rotamer on NBR_ATOM and on the
    # orientation of its bonded atoms, so a central, well-connected atom matters.
    res = L.run_molfile_to_params(sdf, LIGAND_DIR, script=mftp,
                                  resname=LIGAND_RESNAME,
                                  nbr_atom_index=NBR_SD_INDEX)
    LIG_POSE_PDB = res["lig_pdb"]

print()
LIGAND_RESNAME = L.params_resname(LIG_PARAMS)
LIGAND_SPAN = L.ligand_span(LIG_CONFS)
print("residue name      :", LIGAND_RESNAME)
print("lig_conformers.pdb:", L.count_conformers(LIG_CONFS), "rotamers")
print(f"span over rotamers: {LIGAND_SPAN:.1f} A")

# molfile_to_params renames every atom (C1, N1, ... in atom-tree order), so for a
# peptide collapsed into one residue you lose track of which atom was whose CA.
# lig.pdb holds the bound pose with the new names, so the two can be matched on
# coordinates and the identities recovered.
ATOM_MAP = {}
if LIG_POSE_PDB and PEPTIDE_CLEAN_PDB:
    print()
    ATOM_MAP = L.map_params_atoms_to_source(LIG_POSE_PDB, PEPTIDE_CLEAN_PDB)

---
## 6 · Which atom does the path track?

LPF reads `NBR_ATOM` from `lig.params` and follows **that one atom** along the
guide path; Rosetta's `StartFrom(use_nbr=1)` places it on each target point, and
every conformer in the library is overlaid on it.

For a peptide this is the single most consequential choice in the set-up.
`molfile_to_params` tends to pick a terminal atom, and tracking a dangling
terminus would make the guide path describe the motion of one tip rather than
the peptide leaving. It is already set from the central-atom suggestion above;
this cell lets you override it, including by naming an atom of the original
peptide.

Changing it also changes `NBR_RADIUS`, which must enclose the whole residue as
seen from the new atom — too small and Rosetta's neighbour detection silently
truncates interactions. The helper recomputes it over **all** conformers and
adds a margin.

In [ ]:
#@title Tracked atom {run: "auto"}
# params atom name, e.g. C17; empty = keep the current one
OVERRIDE_NBR_ATOM = ""  #@param {type:"string"}
# or name it in the original peptide, e.g. "CA 5"
OVERRIDE_NBR_BY_SOURCE = ""  #@param {type:"string"}

if OVERRIDE_NBR_BY_SOURCE.strip():
    want = OVERRIDE_NBR_BY_SOURCE.split()
    if len(want) != 2:
        raise ValueError('OVERRIDE_NBR_BY_SOURCE must be "<atom> <resseq>", e.g. "CA 5"')
    atom_name, resseq = want[0], int(want[1])
    hits = [p for p, (rn, rs, orig) in ATOM_MAP.items()
            if orig == atom_name and rs == resseq]
    if not hits:
        raise ValueError(f"no params atom maps to {atom_name} of residue {resseq}. "
                         f"Available for that residue: "
                         f"{sorted(o for _, (rn, rs, o) in ATOM_MAP.items() if rs == resseq)}")
    OVERRIDE_NBR_ATOM = hits[0]
    print(f"{OVERRIDE_NBR_BY_SOURCE!r} -> params atom {OVERRIDE_NBR_ATOM}\n")

print("current NBR_ATOM  :", L.params_nbr_atom(LIG_PARAMS))
print("current NBR_RADIUS:", L.params_nbr_radius(LIG_PARAMS))
print()
ranked, scores = L.suggest_nbr_atom(conformers_pdb=LIG_CONFS,
                                    params_path=LIG_PARAMS, top=8)

if OVERRIDE_NBR_ATOM.strip():
    print()
    L.set_nbr_atom(LIG_PARAMS, OVERRIDE_NBR_ATOM.strip(),
                   conformers_pdb=LIG_CONFS, margin=0.5)

NBR_ATOM = L.params_nbr_atom(LIG_PARAMS)
print()
print("tracking:", (L.describe_atom(NBR_ATOM, ATOM_MAP) if ATOM_MAP else NBR_ATOM))

---
## 7 · Assemble `start.pdb`

The layout LPF requires, and why:

* **the ligand comes last** — `PDBHandler.prepare_start` streams protein atoms
  and non-ligand HETATMs into the protein block only until it first meets the
  ligand chain, so anything after the ligand is dropped;
* **the ligand is HETATM, one chain, one residue** — `read_window` ignores ATOM
  records, and every constraint is written as `<atom> <resnum><chain>`;
* **its atom names match `lig.params`** — Rosetta matches ligand atoms to the
  residue type by name. File *order* need not match (the upstream demo's own
  `start.pdb` is a permutation of its params order).

The ligand block is taken from the `lig.pdb` that `molfile_to_params` wrote: its
coordinates are the bound pose and its names are already Rosetta's, which
removes the manual PyMOL renaming in `demo/tutorial.md` 1.2.4.

`header.pdb` is prepended verbatim to every generated `task.pdb` — used for
metal-binding constraints, membrane spans or symmetry records. Empty otherwise.

### Relaxing the start pose (tutorial section 1.3)

LPF assumes the start pose is already relaxed into the Rosetta force field. The
tutorial does this with the standalone `minimize` application and recommends
several attempts, keeping the best. `MINIMIZE_START` does the equivalent through
the `rosetta_scripts` binary you already configured, so it works on either
backend.

This is **not** the same as window 0 of the trajectory: that pass
(`zero_minimize.xml`) only settles the interface region under the `LigandArea`
movemap. An unrelaxed complex shows up as a large window-0 energy.

The tutorial's `minimize` command is unrestrained; `MINIMIZE_CA_SD` adds
C-alpha coordinate restraints so the fold cannot drift, which a free all-atom
minimisation of a whole protein can cause. Set it to 0 to reproduce the
tutorial exactly. Either way, **look at the result** — the tutorial calls manual
verification highly recommended, and this is the step most likely to need
system-specific attention (metalloenzymes, non-standard protonation, structural
waters).

In [ ]:
#@title start.pdb {run: "auto"}
# residue number of the ligand in its own chain
LIGAND_RESNUM = 1  #@param {type:"integer"}
# lines prepended to every task.pdb (metal constraints, membrane spans); normally empty
HEADER_LINES = ""  #@param {type:"string"}
# relax the complex into the Rosetta force field first (tutorial 1.3)
MINIMIZE_START = False  #@param {type:"boolean"}
# attempts; the best by constraint-free total_score is kept
MINIMIZE_NSTRUCT = 5  #@param {type:"integer"}
# C-alpha restraint sd; 0 = the tutorial's unrestrained minimize
MINIMIZE_CA_SD = 0.5  #@param {type:"number"}

pose = LIG_POSE_PDB
if pose is None:
    raise ValueError(
        "No ligand pose file. Either use an input mode that produces lig.pdb, or "
        "set EXISTING_LIGAND_POSE to a PDB of the ligand in its bound pose whose "
        "atom names match lig.params.")
START_PDB = L.assemble_start_pdb(
    PROTEIN_PDB, pose, os.path.join(PROJECT_DIR, "start.pdb"),
    params_path=LIG_PARAMS, ligand_chain=LIGAND_CHAIN,
    ligand_resnum=LIGAND_RESNUM)


HEADER_PDB = L.write_header_pdb(
    os.path.join(PROJECT_DIR, "header.pdb"),
    [l for l in HEADER_LINES.split("\n") if l.strip()])
print()
ok = L.check_start_pdb_vs_params(START_PDB, LIG_PARAMS, LIGAND_CHAIN)
if not ok:
    raise ValueError("start.pdb does not match lig.params — see above. LPF would "
                     "either crash or silently score the wrong thing.")
print()
L.pdb_summary(START_PDB, "start.pdb:")

if MINIMIZE_START:
    print()
    print("=" * 64)
    print("relaxing the start pose into the Rosetta force field (tutorial 1.3)")
    print("=" * 64)
    # the minimisation reads the options file, which section 9 writes; put an
    # options file and VRT1.params in place now so this can run here
    L.write_protocol_xmls(FILES_DIR, XML_SRC, chain=LIGAND_CHAIN,
                          tuning=L.LIGAND_XML_TUNING, files=("coarsepath.xml",),
                          verbose=False)
    L.write_options_file(os.path.join(FILES_DIR, "options"),
                         database=ROSETTA_DATABASE or None)
    print()
    START_PDB = L.minimize_start_pose(
        START_PDB, os.path.join(PROJECT_DIR, "start_minimized.pdb"),
        rosetta=ROSETTA_SCRIPTS, files_dir=FILES_DIR, ligand_dir=LIGAND_DIR,
        ligand_chain=LIGAND_CHAIN, nstruct=MINIMIZE_NSTRUCT,
        ca_restraint_sd=MINIMIZE_CA_SD, n_proc=N_PROC)
print()
NBR_ATOM = L.params_nbr_atom(LIG_PARAMS)
START_NBR = [a.xyz for a in L.read_pdb_atoms(START_PDB)
             if a.chain == LIGAND_CHAIN and a.name == NBR_ATOM][0]
print(f"tracked atom {NBR_ATOM} starts at "
      f"[{START_NBR[0]:.3f} {START_NBR[1]:.3f} {START_NBR[2]:.3f}]")
print()
L.conformer_subsampling_preview(START_PDB, LIG_CONFS, LIGAND_CHAIN, LIG_PARAMS)

---
## 8 · Where does it leave? (the two guide points)

The coarse stage takes exactly two points: the tracked atom's position, and a
point roughly outside the protein. It only reads the first and last point of the
file — everything between is generated by bisection — and it jitters the
endpoint anyway (`endpoint_jitter`), so the direction only has to be roughly
right.

Rather than reading coordinates off PyMOL, `scan` casts rays from the bound pose
and ranks them by how wide the exit is: each ray is sampled every 0.5 Å, the
clearance at each sample is the distance to the nearest protein heavy atom, and
the ray's **bottleneck** is the narrowest clearance before it reaches open
solvent or gets pinched shut. Compare that bottleneck with your molecule's size.

Use `manual` when you know the route, and `use_file` to skip stage 1 entirely
with a path from CAVER or AQUA-DUCT (that path needs ≥ 4 points for the
trajectory stage's spline, and must run **from the binding site outwards** —
LPF overwrites its first point with the ligand's real position, so a tunnel
exported outside-in is silently mangled rather than rejected; the cell below
checks for that).

> Clearance is measured from the **tracked atom**, not the whole molecule. For
> an elongated peptide the far end is still buried
> when the tracked atom first reaches solvent, so the endpoint distance is set
> from the molecule's span by default.

In [ ]:
#@title Exit direction {run: "auto"}
# how the exit direction is chosen
EXIT_MODE = "scan"  #@param ["scan", "manual", "use_file"]
# mode 'scan': which ranked direction to take
RAY_INDEX = 0  #@param {type:"integer"}
# mode 'scan': restrict to a cone, e.g. '0,1,0'
HINT_VECTOR = ""  #@param {type:"string"}
HINT_CONE_DEG = 60.0  #@param {type:"number"}
# mode 'manual': 'x,y,z' of the exit point
EXIT_POINT = ""  #@param {type:"string"}
# mode 'use_file': a CAVER/AQUA-DUCT path, site-to-outside (xyz text or PDB)
GUIDE_PATH_FILE = ""  #@param {type:"string"}
# mode 'use_file': flip a tunnel exported outside-in
REVERSE_PATH = False  #@param {type:"boolean"}
# 0 = auto from the molecule's span
MIN_EXIT_DISTANCE = 0.0  #@param {type:"number"}
# how far from the protein the endpoint must sit
EXIT_CLEARANCE = 8.0  #@param {type:"number"}
N_SCAN_DIRECTIONS = 512  #@param {type:"integer"}

import numpy as np
PROTEIN_COORDS = L.protein_heavy_coords(START_PDB, exclude_chain=LIGAND_CHAIN)
min_dist = MIN_EXIT_DISTANCE or max(15.0, LIGAND_SPAN / 2 + 10.0)
if not MIN_EXIT_DISTANCE:
    print(f"MIN_EXIT_DISTANCE auto: {min_dist:.0f} A "
          f"(span {LIGAND_SPAN:.1f} A / 2 + 10 A, floor 15 A)\n")

RAYS = None
if EXIT_MODE == "use_file":
    GUIDE_PATH = os.path.abspath(os.path.expanduser(GUIDE_PATH_FILE))
    if not os.path.isfile(GUIDE_PATH):
        raise FileNotFoundError(GUIDE_PATH)
    pts = L.read_guide_points(GUIDE_PATH)
    print(f"using your path: {len(pts)} points")
    print()
    # A tunnel finder does not guarantee an orientation, and LPF overwrites point 0
    # with the ligand's real position — so a path exported outside-in is silently
    # mangled rather than rejected.
    pts, was_reversed = L.check_path_direction(pts, START_NBR,
                                               auto_reverse=REVERSE_PATH)
    if was_reversed and not REVERSE_PATH:
        raise ValueError("the supplied path runs the wrong way (see above). Set "
                         "REVERSE_PATH=True to flip it, or export it site-to-outside.")
    if was_reversed:
        GUIDE_PATH = L.write_path_points(os.path.join(PROJECT_DIR, "path_reversed"), pts)
    print()
    L.path_clearance_report(pts, PROTEIN_COORDS, ligand_radius=LIGAND_SPAN / 2)

elif EXIT_MODE == "manual":
    end = np.array([float(v) for v in EXIT_POINT.split(",")])
    GUIDE_PATH = L.write_two_point_path(os.path.join(PROJECT_DIR, "path"),
                                        START_NBR, end)
else:
    hint = ([float(v) for v in HINT_VECTOR.split(",")] if HINT_VECTOR.strip() else None)
    RAYS = L.exit_direction_scan(
        PROTEIN_COORDS, START_NBR, n_dirs=N_SCAN_DIRECTIONS,
        hint=hint, hint_cone_deg=HINT_CONE_DEG if hint else None, top=10)
    ray = RAYS[int(RAY_INDEX)]
    print(f"\nchosen ray #{RAY_INDEX}: {ray}")
    if ray.bottleneck < LIGAND_SPAN / 4:
        print(f"[warn] bottleneck {ray.bottleneck:.1f} A is tight for a molecule "
              f"spanning {LIGAND_SPAN:.1f} A — the protein will have to open up. "
              f"That is legitimate (and what memory_mode is for) but it makes the "
              f"run harder; try another RAY_INDEX if one is wider.")
    print()
    end = L.exit_point_along(PROTEIN_COORDS, START_NBR, ray.direction,
                             clearance=EXIT_CLEARANCE, min_dist=min_dist,
                             ligand_span=LIGAND_SPAN)
    GUIDE_PATH = L.write_two_point_path(os.path.join(PROJECT_DIR, "path"),
                                        START_NBR, end)

In [ ]:
# 3D check: protein surface, the peptide, and the proposed exit vector
try:
    import py3Dmol
    v = py3Dmol.view(width=820, height=520)
    v.addModel(open(START_PDB).read(), "pdb")
    v.setStyle({"cartoon": {"color": "lightgrey"}})
    v.setStyle({"chain": LIGAND_CHAIN}, {"stick": {"radius": 0.2, "colorscheme": "cyanCarbon"}})
    v.addSphere({"center": {"x": float(START_NBR[0]), "y": float(START_NBR[1]),
                            "z": float(START_NBR[2])},
                 "radius": 0.8, "color": "blue"})
    if EXIT_MODE != "use_file":
        v.addArrow({"start": {"x": float(START_NBR[0]), "y": float(START_NBR[1]),
                              "z": float(START_NBR[2])},
                    "end": {"x": float(end[0]), "y": float(end[1]), "z": float(end[2])},
                    "radius": 0.25, "color": "red"})
        print("blue sphere = tracked atom (path start), red arrow = exit direction")
    v.addSurface("VDW", {"opacity": 0.55, "color": "white"},
                 {"chain": LIGAND_CHAIN, "invert": 1})
    v.zoomTo({"chain": LIGAND_CHAIN})
    v.show()
except ImportError:
    print("py3Dmol not installed — skipping the 3D view (pip install py3Dmol)")

---
## 9 · RosettaScripts protocols and the options file

Copies the stock protocols out of `rosettaXML_files/` and retargets them, rather
than shipping forks that would drift from the repository.

### What the peptide tuning changes, and why

| change | from → to | reason |
|---|---|---|
| `SCORINGGRIDS width` | 20 → sized from the molecule | the grid is centred on the ligand and nothing outside it is scored; 20 Å does not contain a peptide |
| `LigandArea cutoff` | 6/6/7 → 8/8/10 Å | a peptide contacts far more of the protein, and only residues inside these cutoffs may repack or move |
| `Calpha_restraints` (main) | 0.3 → 0.5 | this is a standard deviation, so a larger value lets the backbone actually open; left at 0.1 in `zero_minimize.xml`, which only settles the start pose |
| `DetectProteinLigandInterface` cuts | 6/6/6/6 → 7/9/11/13 Å | ditto, for the repacking shell |
| `Transform box_size` | 5 → 8 Å | a bigger body needs a bigger translation box |
| `Transform cycles/repeats` | 500/1 → 500/2 | more sampling per window |
| `HighResDocker cycles` | 9 → 15 | ditto |
| `initial_angle_perturb` (coarse) | 90 → 20° | spinning an elongated peptide freely inside a channel just makes clashes |
| `add_nbr_radius` (coarse) | true → false | the coarse protocol adds `NBR_RADIUS` to its cutoff; that is ~2 Å for a small ligand and 20+ Å for a peptide, which would drag the whole protein into every repack |

The coarse protocol's cycle counts are left alone — stage 1 is meant to be cheap.
`main_rigid.xml` and `main_extended.xml` are not written, because blanket
retuning would erase the only thing that distinguishes them from
`main_standard.xml`.

The options file is written fresh so that `-in:path:database` can be included
when your binary sits outside its source tree — without it every window fails
identically.

In [ ]:
#@title Protocols {run: "auto"}
# 0 = size it from the molecule's span
GRID_WIDTH = 0.0  #@param {type:"number"}
# Transform translation box; also feeds the grid size
TRANSFORM_BOX_SIZE = 8.0  #@param {type:"number"}
# cofactor .params paths, comma-separated
EXTRA_PARAMS_FILES = ""  #@param {type:"string"}
# adds -in:auto_setup_metals
AUTO_SETUP_METALS = False  #@param {type:"boolean"}
# peptide-scaled protocol tuning
APPLY_TUNING = True  #@param {type:"boolean"}

extra = [os.path.abspath(os.path.expanduser(p.strip()))
         for p in EXTRA_PARAMS_FILES.split(",") if p.strip()]
grid_w = GRID_WIDTH or L.auto_grid_width(LIGAND_SPAN, box_size=TRANSFORM_BOX_SIZE)

tuning = L.PEPTIDE_XML_TUNING if APPLY_TUNING else L.LIGAND_XML_TUNING
if APPLY_TUNING:
    tuning = dict(tuning)
    tuning["rules"] = list(tuning["rules"]) + [
        {"tag": "Transform", "name": "transform",
         "attrs": {"box_size": str(int(TRANSFORM_BOX_SIZE))}, "files": L._MAIN_LIKE},
        {"tag": "Transform", "name": "transform_2",
         "attrs": {"box_size": str(int(TRANSFORM_BOX_SIZE))}, "files": L._MAIN_LIKE},
    ]
print()
PROTOCOLS = L.write_protocol_xmls(FILES_DIR, XML_SRC, chain=LIGAND_CHAIN,
                                  tuning=tuning, grid_width=grid_w)
print()
OPTIONS = L.write_options_file(
    os.path.join(FILES_DIR, "options"),
    extra_res=["lig.params", "VRT1.params"] + [os.path.basename(p) for p in extra],
    database=ROSETTA_DATABASE or None,
    auto_setup_metals=AUTO_SETUP_METALS)
EXTRA_FILES = extra

---
## 10 · Stage 1 — coarse path

Recursive bisection between the two guide points: Rosetta places the
peptide at the midpoint under the plane / in-plane /
angle constraints, the midpoint moves to wherever the tracked atom actually
ended up, and the halves are subdivided until neighbouring points are closer
than `step`. Output: **`coarsepath.txt`**.

It is stochastic and not guaranteed to converge — it depends on the endpoint you
chose. Several replicates with `endpoint_jitter` on, then the medoid, is the
recommended practice.

Skip this entire section if you set `EXIT_MODE='use_file'` above.

### Peptide coarse defaults — what differs from the paper's values and why

| key | ligand | peptide | reasoning |
|---|---|---|---|
| `step` | 2.0 | 3.0 Å | each node is a full Rosetta run and a peptide's node is much dearer; 2 Å buys no extra routing detail |
| `memory_mode` | false | **true** | with it off, every node restarts from the original protein, so the channel never opens and a peptide-sized body may find no route at all. The cost is that induced distortion accumulates along a stochastic run — which is itself a reason to run replicates |
| `conformer_subsampling` | false | **true** (4.0 Å) | the full library is large and mostly irrelevant to the current pose; the threshold is generous so the ladder stays connected |
| `rmsd_constraint_sd` | 1.5 | 2.0 Å | the effective weight is `sd·√N`, i.e. already normalised for atom count, so this really is a looser shape restraint: the peptide has to change conformation to get out |
| `plane_constraint_point_extent` | 7 | 12 Å | circumradius of the virtual triangle defining the guide plane; it must exceed the molecule or the three dihedrals holding it in-plane are badly conditioned |
| `angle_constraint_bound` | 2.25 | 2.00 rad | an elongated body cannot keep its tracked atom as straight |

In [ ]:
#@title Stage 1 parameters {run: "auto"}
# dock the ligand onto the path's first point before stage 1
DOCK_ZERO_COARSE = False  #@param {type:"boolean"}
# the paper's >50-atom rule (conformer_rmsd_threshold 5.0)
APPLY_LARGE_LIGAND_RULE_COARSE = True  #@param {type:"boolean"}
# bisection stops when neighbouring nodes are closer than this
COARSE_STEP = 3.0  #@param {type:"number"}
# Rosetta decoys per node
COARSE_NSTRUCT = 10  #@param {type:"integer"}
# independent coarse runs; the medoid path goes forward
N_REPLICATES = 1  #@param {type:"integer"}
# each replicate lands at a slightly different endpoint
ENDPOINT_JITTER_RADIUS = 5.0  #@param {type:"number"}
# cost estimate only; the paper's benchmark average — recalibrate from your first node
SEC_PER_DECOY_GUESS = 16.0  #@param {type:"number"}
# the coarse stage cannot resume, so re-runs need a clean directory
CLEAN_PREVIOUS = True  #@param {type:"boolean"}

coarse_params = dict(L.PEPTIDE_COARSE_PARAMS)
if APPLY_LARGE_LIGAND_RULE_COARSE:
    coarse_params = L.apply_large_ligand_rule(
        coarse_params, len(L.params_atom_names(LIG_PARAMS)))
    print()
coarse_params["step"] = COARSE_STEP
coarse_params["dock_zero"] = DOCK_ZERO_COARSE
coarse_params["endpoint_jitter"] = N_REPLICATES > 1 or coarse_params["endpoint_jitter"]
coarse_params["endpoint_jitter_radius"] = ENDPOINT_JITTER_RADIUS

print("coarse parameters:")
for k, v in sorted(coarse_params.items()):
    print(f"   {k:34s} {v}")

_gp = L.read_guide_points(GUIDE_PATH)
if not L.check_path_anchor(_gp, START_NBR, dock_zero=coarse_params.get("dock_zero")):
    raise ValueError("the guide path and the start pose disagree (see above). "
                     "Set DOCK_ZERO_COARSE=True, or fix the path / start pose.")
print()
guide_len = float(np.linalg.norm(_gp[-1] - _gp[0])) \
    if EXIT_MODE != "use_file" else 0.0
n_nodes = max(1, int(guide_len / COARSE_STEP)) if guide_len else 15
print(f"\nguide segment {guide_len:.1f} A -> roughly {n_nodes} nodes per replicate")
L.estimate_cost(n_nodes, COARSE_NSTRUCT, SEC_PER_DECOY_GUESS, N_PROC, label="node")
if N_REPLICATES > 1:
    print(f"x {N_REPLICATES} replicates")

In [ ]:
# One coarse run per replicate, each in its own directory.
COARSE_RUNS, COARSE_PATHS = [], []
if EXIT_MODE == "use_file":
    print("EXIT_MODE='use_file' — skipping stage 1; your path goes straight to stage 2.")
else:
    for rep in range(int(N_REPLICATES)):
        run_dir = os.path.join(PROJECT_DIR, f"coarse_{rep:02d}")
        L.prepare_run_dir(run_dir, clean=CLEAN_PREVIOUS)
        cfg = L.build_run_config(
            "coarse", ROSETTA_SCRIPTS, FILES_DIR, LIGAND_DIR, GUIDE_PATH, run_dir,
            start_pdb=START_PDB, header_pdb=HEADER_PDB, ligand_chain=LIGAND_CHAIN,
            params=coarse_params, nstruct=COARSE_NSTRUCT, n_proc=N_PROC,
            extra_files=EXTRA_FILES, rosetta_database=ROSETTA_DATABASE or None,
            archive=ARCHIVE_WINDOWS)
        cfg_path = L.write_run_config(os.path.join(run_dir, "run.json"), cfg)

        errs, warns = L.preflight_config(cfg_path, "coarse")
        if errs:
            raise RuntimeError(f"preflight found {len(errs)} problem(s) — fix them above")

        print(f"\n{'=' * 64}\nreplicate {rep + 1}/{N_REPLICATES}\n{'=' * 64}")
        L.run_lpf_stage("coarse", LPF_REPO, cfg_path, run_dir,
                        log_path=os.path.join(run_dir, "coarse.log"))
        COARSE_RUNS.append(run_dir)
        COARSE_PATHS.append(os.path.join(run_dir, "coarsepath.txt"))

In [ ]:
# Inspect the coarse result(s) and choose the path for stage 2.
if EXIT_MODE == "use_file":
    print("using your supplied path:", GUIDE_PATH)
    TRAJ_PATH = GUIDE_PATH
else:
    for rd in COARSE_RUNS:
        print("=" * 64)
        L.summarize_run(rd, stage="coarse")
        print()
    TRAJ_PATH, k, _D = (L.coarse_path_medoid(COARSE_PATHS) if len(COARSE_PATHS) > 1
                        else (COARSE_PATHS[0], 0, None))
    print("\npath going forward to stage 2:", TRAJ_PATH)

pts = L.read_guide_points(TRAJ_PATH)
print()
L.path_clearance_report(pts, PROTEIN_COORDS, ligand_radius=LIGAND_SPAN / 2)
if len(pts) < 4:
    raise ValueError(
        f"the path has only {len(pts)} points. The trajectory stage fits a cubic "
        f"smoothing spline and needs at least 4 — lower COARSE_STEP, or supply a "
        f"denser path.")

try:
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(6.4, 5))
    ax = fig.add_subplot(111, projection="3d")
    for i, p in enumerate(COARSE_PATHS):
        q = np.loadtxt(p)
        ax.plot(*q.T, "-o", ms=2.5, lw=1.0, alpha=0.9 if p == TRAJ_PATH else 0.35,
                label=f"replicate {i}" + (" (medoid)" if p == TRAJ_PATH else ""))
    ax.scatter(*np.atleast_2d(START_NBR).T, c="k", s=60, label="bound pose")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("coarse path(s)")
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
except Exception as exc:
    print("(3D plot skipped:", exc, ")")

---
## 11 · Stage 2 — trajectory

The guide path is fitted with a smoothing spline, resampled at `step` spacing,
and one Rosetta run is done per window, each starting from the previous window's
result. Output: **`interface_scores.txt`** — the `interface_delta_X` binding
energy per window, which is the result you analyse.


> **Colab sessions end.** `checkpoint_interval` is set low and the run resumes
> automatically from `checkpoint.pkl` — just re-run the cell after reconnecting
> (and keep `PROJECT_DIR` on Drive). `FRESH_START=True` overrides and begins again.

### Peptide trajectory defaults — what differs and why

| key | ligand | peptide | reasoning |
|---|---|---|---|
| `step` | 0.5 | 1.0 Å | halves the window count, and each peptide window is already several times dearer. Drop it back towards 0.5 if the conformation jumps between windows |
| `smooth` | 0.5 | 1.0 | a long body cannot take sharp corners; the script prints the max turning angle and what 2× / 5× smoothing would give |
| `conformer_rmsd_threshold` | 1.5 | 3.0 Å | a peptide's conformers differ far more; 1.5 Å would leave it near-rigid |
| `rmsd_constraint_sd` | 0.75 | 1.25 Å | looser shape restraint (the weight is already `sd·√N`) |
| `rmsd_constraint_hydrogens` | true | **false** | hydrogens roughly double the atom count and add stiffness without shape information |
| `ipdc_d0` / `ipdc_sd` | 3.0 / 0.5 | 4.0 / 0.75 Å | the tracked atom of a long molecule legitimately sits further off the guide point while the rest threads through |
| `end_padding` | 5 | 10 Å | more room to disengage; the in-plane restraint relaxes over the last `end_padding/step` windows |
| `angle_constraint_bound` | 2.00 | 1.80 rad | ditto, direction restraints |
| `next_angle_constraint_bound` | 1.57 | 1.40 rad | ditto |
| `outlier_rmsd` | 2.0 | 3.0 Å | 2.0 Å would reject most decoys of a large, genuinely mobile body |

In [ ]:
#@title Stage 2 parameters {run: "auto"}
# 'benchmark' = the settings from the paper's 136-system benchmark
PARAM_PRESET = "auto"  #@param ["auto", "benchmark"]
# the paper's >50-atom rule: conformer_rmsd_threshold and outlier_rmsd both 5.0
APPLY_LARGE_LIGAND_RULE = True  #@param {type:"boolean"}
# window spacing in A
TRAJ_STEP = 1.0  #@param {type:"number"}
# spline smoothing; larger = smoother, less faithful
TRAJ_SMOOTH = 1.0  #@param {type:"number"}
# decoys per window
TRAJ_NSTRUCT = 10  #@param {type:"integer"}
# second independent scoring pass per window = error bars
RUN_STATS = False  #@param {type:"boolean"}
NSTRUCT_STATS = 10  #@param {type:"integer"}
# dock onto the path's first point before starting (ligand present but unplaced)
DOCK_ZERO = False  #@param {type:"boolean"}
# 0 disables; the run resumes from checkpoint.pkl
CHECKPOINT_INTERVAL = 5  #@param {type:"integer"}
# run each window on a RAM disc, copy results back
USE_TMP = False  #@param {type:"boolean"}
# ignore an existing checkpoint
FRESH_START = False  #@param {type:"boolean"}
# cost estimate only; the paper's benchmark average — recalibrate from your first window
SEC_PER_DECOY_GUESS = 16.0  #@param {type:"number"}

traj_params = dict(L.BENCHMARK_TRAJ_PARAMS if PARAM_PRESET == "benchmark"
                   else L.PEPTIDE_TRAJ_PARAMS)
if PARAM_PRESET == "benchmark":
    print("using the paper's benchmark settings (SI, Geometry of Tunnels): "
          "smooth 5.0,\nrmsd sd 0.66, ipdc d0 2.0, angle 2.5/0.5 — these suit an "
          "external CAVER path.\n")
# The paper loosens two keys for ligands over 50 atoms: conformer_rmsd_threshold
# and outlier_rmsd both to 5.0 (SI, benchmark settings). Every peptide trips this.
if APPLY_LARGE_LIGAND_RULE:
    traj_params = L.apply_large_ligand_rule(
        traj_params, len(L.params_atom_names(LIG_PARAMS)))
    print()
traj_params["step"] = TRAJ_STEP
traj_params["smooth"] = TRAJ_SMOOTH
traj_params["run_stats"] = RUN_STATS
traj_params["dock_zero"] = DOCK_ZERO

print("trajectory parameters:")
for k, v in sorted(traj_params.items()):
    print(f"   {k:34s} {v}")

pts = L.read_guide_points(TRAJ_PATH)
# Unless dock_zero is on, LPF moves the path's first point onto the ligand. If the
# path starts somewhere else entirely that leaves a huge first segment and a
# meaningless spline, so check the two agree before paying for a run.
if not L.check_path_anchor(pts, START_NBR, dock_zero=traj_params.get("dock_zero")):
    raise ValueError("the guide path and the start pose disagree (see above). "
                     "Set DOCK_ZERO=True, or fix the path / start pose.")
print()
path_len = float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum())
n_windows = max(1, int(path_len / TRAJ_STEP))
print(f"\nguide path {path_len:.1f} A -> roughly {n_windows} windows at "
      f"step={TRAJ_STEP} A")
L.estimate_cost(n_windows, TRAJ_NSTRUCT, SEC_PER_DECOY_GUESS, N_PROC)
if RUN_STATS:
    print("plus the statistics pass:")
    L.estimate_cost(n_windows, NSTRUCT_STATS, SEC_PER_DECOY_GUESS, N_PROC)

In [ ]:
TRAJ_RUN = os.path.join(PROJECT_DIR, "trajectory")
L.prepare_run_dir(TRAJ_RUN, clean=False)   # keep the checkpoint so a stopped run resumes

cfg = L.build_run_config(
    "trajectory", ROSETTA_SCRIPTS, FILES_DIR, LIGAND_DIR, TRAJ_PATH, TRAJ_RUN,
    start_pdb=START_PDB, header_pdb=HEADER_PDB, ligand_chain=LIGAND_CHAIN,
    params=traj_params, nstruct=TRAJ_NSTRUCT, nstruct_stats=NSTRUCT_STATS,
    n_proc=N_PROC, extra_files=EXTRA_FILES,
    rosetta_database=ROSETTA_DATABASE or None, archive=ARCHIVE_WINDOWS,
    checkpoint_interval=CHECKPOINT_INTERVAL, use_tmp=USE_TMP)
cfg_path = L.write_run_config(os.path.join(TRAJ_RUN, "run.json"), cfg)

errs, warns = L.preflight_config(cfg_path, "trajectory")
if errs:
    raise RuntimeError(f"preflight found {len(errs)} problem(s) — fix them above")

In [ ]:
# The long one. Progress is streamed; each window prints its constraint energies
# and how far the tracked atom ended up from the guide plane (d_to_plane) and the
# guide point (d_in_plane). Small values mean the constraints are holding.
L.run_lpf_stage("trajectory", LPF_REPO, cfg_path, TRAJ_RUN,
                fresh=FRESH_START,
                log_path=os.path.join(TRAJ_RUN, "trajectory.log"))

---
## 12 · Results

| file | what it is |
|---|---|
| **`interface_scores.txt`** | `interface_delta_X` per scored window — **the main result** |
| `nbr_trajectory.txt` | tracked atom's position per window |
| `full_trajectory.pdb` | protein + peptide along the path, one frame per window |
| `lig_trajectory.pdb` | the peptide alone |
| `total_scores.txt` | whole-complex energy (mostly for debugging) |
| `stats.pkl` | per-window score distributions, if `run_stats` was on |
| `guide_path`, `spline`, `normals`, `input_path` | path construction intermediates |

Constraint energies are subtracted before any decoy is ranked or reported, so
these numbers are not contaminated by the restraints that produced them.

**One replicate is noisy.** Read the shape, not the digits; pool replicates
before quoting numbers (appendix).

In [ ]:
print("=" * 64)
info = L.summarize_run(TRAJ_RUN, stage="trajectory")

In [ ]:
import matplotlib.pyplot as plt
ax = L.plot_energy_profile(TRAJ_RUN, x="distance", show_stats=True,
                           smooth_window=0)
plt.show()

d = L.load_run_scores(TRAJ_RUN)
y = d["interface_scores"]
if y is not None and len(y):
    bound, free = float(y[0]), float(y[-1])
    print(f"window 0 (bound)   {bound:+.2f} REU")
    print(f"last window (free) {free:+.2f} REU")
    print(f"deepest minimum    {y.min():+.2f} REU at window {int(y.argmin())}")
    print(f"energy released along this path: {free - y.min():+.2f} REU")
    print()
    print("Local minima along the way are where the " + "peptide" +
          " pauses against")
    print("gateway residues. To find out which residues, look at the window PDBs:")
    print(f"  {TRAJ_RUN}/window_XXXX/best.pdb   (zipped if archive was on)")

In [ ]:
# Overlay: the protein once, every pose of the peptide coloured blue (bound)
# to red (released). mode="frames" animates instead.
try:
    L.view_trajectory(TRAJ_RUN, ligand_chain=LIGAND_CHAIN, mode="overlay", stride=1)
except ImportError:
    print("py3Dmol not installed (pip install py3Dmol)")
except Exception as exc:
    print("3D view unavailable:", exc)

### Pooling replicates: the paper's clustering

One replicate is noisy; the paper's results come from an ensemble. Its recipe
(SI, *Clustering of sampled paths*) is implemented here:

1. the distance between two trajectories is the **frame-by-frame coordinate RMSD
   of the ligand**, summed over frames with **linearly decreasing weights** — so
   differences while the ligand is still inside the tunnel count for more than
   differences out in bulk solvent;
2. where one path is shorter, each missing frame is charged a fixed **0.5 Å**;
3. the **negative** of that matrix goes to **affinity propagation**
   (scikit-learn, default parameters) as a precomputed affinity;
4. the interface profile is **averaged within each cluster**;
5. clusters of **10 or fewer** paths are discarded;
6. the most populated cluster is drawn bold, the rest thin and semi-transparent.

This needs several finished trajectory runs — the paper pooled about 30. Produce
them by re-running section 11 with a different `RUN_NAME`, or on a cluster with
the array job in the appendix. With only a handful, lower `MIN_CLUSTER_SIZE` or
nothing will survive step 5.

In [ ]:
#@title Cluster pooled replicates {run: "auto"}
# finished trajectory runs, or a glob; empty = repl-*/ and trajectory*/ in the project
REPLICATE_GLOB = ""  #@param {type:"string"}
# discard clusters with this many members or fewer (10 is the paper's value)
MIN_CLUSTER_SIZE = 10  #@param {type:"integer"}
# distance charged per missing frame (0.5 is the paper's value)
MISSING_PENALTY = 0.5  #@param {type:"number"}
import glob as _glob
runs = sorted(_glob.glob(os.path.expanduser(REPLICATE_GLOB))) if REPLICATE_GLOB \
    else (sorted(_glob.glob(os.path.join(PROJECT_DIR, "repl-*")))
          + sorted(_glob.glob(os.path.join(PROJECT_DIR, "trajectory*"))))
runs = [r for r in runs if os.path.isdir(r)]

if len(runs) < 2:
    print(f"found {len(runs)} run(s) - clustering needs at least two.")
    print("Re-run the trajectory section with a different RUN_NAME, or use the")
    print("cluster array job in the appendix, then come back here.")
else:
    clusters = L.cluster_trajectories(
        runs, ligand_chain=LIGAND_CHAIN, missing_penalty=MISSING_PENALTY,
        min_cluster_size=MIN_CLUSTER_SIZE, plot=True)
    print()
    for c in clusters["kept"]:
        m = clusters["means"][c]
        print(f"  cluster {c}: n={m['n']:3d}  exemplar {m['exemplar']}  "
              f"bound {m['mean'][0]:+.2f} REU  final {m['mean'][-1]:+.2f} REU")
    plt.show()

### If `interface_scores.txt` came out all zeros

That is not a result — it means Rosetta's `InterfaceScoreCalculator` contributed
no `interface_delta_<chain>` and the column was written as a placeholder. The
poses themselves are fine, so the profile can be recomputed without redoing any
docking: each window's selected pose is rescored once with upstream's
scoring-only `zero_score.xml`. One Rosetta call per window instead of `nstruct`
docking runs — minutes, not hours. Archived windows are read out of their zips,
and the original file is left untouched so you can compare.

In [ ]:
#@title Rescore a zeroed run {run: "auto"}
# recompute the interface energies from the saved poses
RUN_RESCORE = False  #@param {type:"boolean"}
# run directory to rescore; empty = the one just run
RESCORE_RUN = ""  #@param {type:"string"}
# replace interface_scores.txt so the plots and clustering use it
INSTALL_RESCORED = True  #@param {type:"boolean"}
if "RESCORE_RUN" not in dir():
    RESCORE_RUN = ""
target = RESCORE_RUN or TRAJ_RUN
_scores = L.load_run_scores(target)["interface_scores"]
if _scores is not None and len(_scores) and not np.allclose(_scores, 0.0):
    print("This run already has real interface energies — nothing to rescore.")
    print(f"  bound {_scores[0]:+.2f} REU, final {_scores[-1]:+.2f} REU")
elif RUN_RESCORE:
    # zero_score.xml is upstream's scoring-only protocol: one
    # InterfaceScoreCalculator and nothing else.
    L.write_protocol_xmls(FILES_DIR, XML_SRC, chain=LIGAND_CHAIN,
                          files=("zero_score.xml",), verbose=False)
    print()
    rescored = L.rescore_windows(
        target, rosetta=ROSETTA_SCRIPTS, files_dir=FILES_DIR,
        ligand_dir=LIGAND_DIR, ligand_chain=LIGAND_CHAIN,
        extra_files=EXTRA_FILES, install=INSTALL_RESCORED)
else:
    print("interface_scores.txt is all zeros, which is not a result.")
    print("Set RUN_RESCORE=True to recompute it from the saved poses — one")
    print("scoring call per window, no re-docking.")

### How many replicates do you need?

Measured rather than guessed. For each ensemble size *n*, bootstrap resamples of
*n* runs are drawn and the mean profile computed for each; the spread of those
means is how much your answer would still move if you had run a different *n*.
To resolve a feature of size *D* REU you want that spread comfortably below *D*.

It falls as 1/√n, drawn as a reference. A curve sitting **above** the reference
means the replicates disagree structurally — different exit routes — and the
answer is to cluster them, not to average harder.

For scale: the paper's tutorial runs **30** replicates in its worked example, and
its clustering discards clusters of 10 or fewer paths, so an ensemble intended
for clustering has to be a good deal larger than 10. The SI does not state the
per-system counts behind the published figures.

In [ ]:
#@title Replicate convergence {run: "auto"}
# replicate run dirs, or a glob; empty = repl-*/ in the project
CONVERGENCE_GLOB = ""  #@param {type:"string"}
import glob as _g
runs = sorted(_g.glob(os.path.expanduser(CONVERGENCE_GLOB))) if CONVERGENCE_GLOB \
    else sorted(_g.glob(os.path.join(PROJECT_DIR, "repl-*")))
runs = [r for r in runs if os.path.isdir(r)]
if len(runs) < 3:
    print(f"found {len(runs)} replicate(s); this needs at least 3.")
    print("Produce them by re-running the trajectory section with a different")
    print("RUN_NAME (repl-00, repl-01, ...), or with the sbatch array in the")
    print("appendix, then come back here.")
else:
    conv = L.replicate_convergence(runs, step=traj_params["step"], plot=True)
    plt.show()

In [ ]:
# Package the analysis files. include_windows=True also takes every window
# directory, which is where the decoy archives are — usually hundreds of MB.
zip_path = L.zip_results(TRAJ_RUN, os.path.join(PROJECT_DIR, "trajectory_results.zip"),
                         include_windows=False)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("(automatic download unavailable:", exc, ")")
    print("The archive is on Drive/disk at the path above.")

---
## Appendix

### Retuning, from `demo/tutorial.md` section 5

**The peptide is too stiff — it moves as a rigid body.**
1. Check the conformer library actually holds distinct conformations.
2. Raise `conformer_rmsd_threshold`.
3. Raise `rmsd_constraint_sd`.
4. Set `rmsd_constraint_hydrogens` false.
5. Lower `angle_constraint_bound`.

**The peptide changes conformation too sharply between windows.** The reverse of
the above, and lower `step`.

**The protein is not flexible enough.**
1. Check `memory_mode` is true.
2. Raise `Calpha_restraints` in `main_standard.xml` (it is a standard deviation,
   so larger is looser).
3. Raise the `LigandArea` cutoffs.

**The protein takes weird conformations.** The reverse of the above.

**A window stalls or the peptide cuts a corner.** The trajectory script prints
the guide path's maximum turning angle at start-up and warns above 25°. Raise
`smooth`, or run more coarse replicates and take a straighter medoid.

### Peptide-specific things that go wrong

**The conformer library is disconnected.** The report in step 5 said `NO`. The
peptide cannot walk from its bound conformation to the extended ones, so it
leaves as a rigid body or not at all. Raise `N_MEDIUM` and `SIGMA_MEDIUM_DEG`,
lower `CONFORMER_PRUNE_RMSD`, or raise `conformer_rmsd_threshold`.

**Only 1–2 rotamers survive subsampling.** `conformer_subsampling_preview` in
step 7 tells you this before you spend anything. Raise
`conformer_rmsd_threshold`, or build a denser library.

**`molfile_to_params` refuses the peptide.** You are past its practical range.
Shorten the peptide, or treat the problem as a protein–protein unbinding
question with a different tool.

**The tracked atom is a terminus.** Check the `tracking:` line in step 6. A
dangling terminus makes the guide path describe the motion of one tip.

**The scoring grid is smaller than the peptide.** Step 9 sizes it from the
molecule's span; if you set `GRID_WIDTH` by hand, keep it above
`span + 2*box_size`.

**Window 0 has a huge energy.** The start pose is not relaxed into the Rosetta
force field. Minimise the complex first (`demo/tutorial.md` 1.3).

### If you are using the PyRosetta backend

**Every window fails identically.** Run the launcher by hand in a window
directory to see the real error:
`cd trajectory/window_0001 && ./pyrosetta_backend/rosetta_scripts.pyrosetta @options -parser:protocol task.xml -in:file:s task.pdb -nstruct 1 -out:path:pdb ./results`.
Set `LPF_SHIM_VERBOSE=1` to keep PyRosetta's own tracer output instead of muting it.

**It is slower than you expected.** The shim runs `nstruct` decoys in one
process by default. Raise `SHIM_JOBS` (step 2) to spread them over worker
processes — each one initialises its own PyRosetta, so budget 1-2 GB of RAM
apiece and do not exceed `nstruct`. `rosetta_scripts.n_proc` must stay at 1;
that path is MPI and the shim does not implement it.

**A protocol works with the binary but not the shim.** The shim builds movers
through PyRosetta's RosettaScripts parser, whose entry points differ between
releases; it tries the known ones in order and prints which it used. If none
work with your build, report what it printed — and fall back to a real binary,
which is the reference implementation.

### Replicates on a cluster

A single replicate is noisy. Pool `interface_scores.txt` across independent
trajectory runs for a converged profile.

`lpf_run.py` (shipped next to this notebook) is the command-line version of
everything above, and is easier to drive from a queue: `prep` writes
`project.json`, and each later stage reads it, so they can be separate jobs.

```bash
#!/bin/bash
#SBATCH -n 1
#SBATCH --cpus-per-task=1
#SBATCH --mem=5G
#SBATCH -t 720
lpf_run.py traj --project /path/to/project     --run-name repl-$SLURM_ARRAY_TASK_ID --traj-nstruct 10
```

```bash
lpf_run.py prep ... --project /path/to/project   # once
lpf_run.py coarse --project /path/to/project --replicates 10
sbatch -J lpf -a 0-29 run.bash                   # 30 trajectory replicates
lpf_run.py cluster --project /path/to/project --convergence
```

On one machine, `lpf_run.py traj --traj-replicates 30` runs them serially
instead. **How many you need** is measurable rather than a guess — the
convergence cell above bootstraps your runs and reports how much the pooled mean
still moves as replicates are added. For scale, the tutorial's worked example
uses 30, and its clustering discards clusters of 10 or fewer paths; the SI does
not give the per-system counts behind the published figures.

Or drive the upstream script directly, with `run.json` one level up:

```bash
RUNDIR=repl-$SLURM_ARRAY_TASK_ID
mkdir -p $RUNDIR && cp run.json $RUNDIR && cd $RUNDIR
python /path/to/LigandPathFinder/lpf/lpf_trajectory.py run.json
```

Then pool them:

```python
import numpy as np, glob
runs = sorted(glob.glob("repl-*/interface_scores.txt"))
profiles = [np.loadtxt(f) for f in runs]
n = min(map(len, profiles))
A = np.stack([p[:n] for p in profiles])
mean, sem = A.mean(0), A.std(0) / np.sqrt(len(A))
```

### Reference and licences

* Method: Zlobin A, Eisenhuth P, Ferrario V, Meiler J. *Exploration of ligand (un)binding trajectories with Rosetta*. bioRxiv 2026. doi:[10.64898/2026.10.07.757317](https://www.biorxiv.org/content/10.64898/2026.10.07.757317v1)
* Code: [`meilerlab/LigandPathFinder`](https://github.com/meilerlab/LigandPathFinder) — MIT, covering the LPF code only.
* Every `run.json` key: `docs/config_reference.md` in the repository. Keys each
  stage ignores: `docs/unused_config_settings.md`.
* Reference runs and the paper's dataset:
  [10.5281/zenodo.23183178](https://doi.org/10.5281/zenodo.23183178).
* **Rosetta is licensed separately** and is not included. Free for academic and
  non-commercial use; commercial use needs its own licence.
  `molfile_to_params.py` and the `rosetta_py` modules this notebook can download
  are Rosetta files, covered by that same licence.

### What this notebook adds on top of the upstream code

Nothing here patches LPF; it all ends up as input files the upstream scripts
read. The non-obvious parts:

* `start.pdb` assembly that enforces LPF's unwritten requirements (ligand last,
  HETATM, one residue, names matching `lig.params`) and checks them;
* protocol XMLs retargeted from `rosettaXML_files/` to your chain and your
  molecule's size, instead of forked copies;
* an options file that can carry `-in:path:database`, for a binary moved out of
  its source tree;
* `pyrosetta_rosetta_scripts.py`, a `rosetta_scripts`-compatible front end so a
  pip-installed PyRosetta can serve as the backend — LPF execs the binary and
  parses `score.sc`, so the shim reproduces that command line and that file
  format rather than patching LPF;
* a preflight that reports every configuration problem at once rather than
  failing on the first one mid-run — including the two that otherwise waste a
  whole run: a guide path with fewer than 4 points (the trajectory spline needs
  4) and a protocol/`ligand_chain` mismatch (Rosetta never reads
  `input.ligand_chain`);
* exit-direction scanning with bottleneck widths, so the guide points are chosen
  against the geometry rather than by eye;
* `NBR_ATOM` re-selection with a recomputed `NBR_RADIUS`;
* conformer-library diagnostics in LPF's own RMSD metric — how many rotamers
  survive subsampling, and whether the library is connected.